In [ ]:
!pip install biopython

from Bio import Entrez
from time import sleep
import os
from google.colab import drive

# --- Configuration ---

Entrez.email = "junghosohn8@gmail.com"
INTERVAL_YEARS = 5
START_YEAR = 1980
END_YEAR = 2025
MAX_RECORDS = 10000
RETMAX = 10000
SLEEP_TIME = 0.34

QUERY_BASE = '("P-glycoprotein" OR "P-gp" OR "ABCB1") AND (inhibit* OR substrate OR induc*) AND hasabstract[text] AND english[lang] AND (human)'
OUTPUT_DIR = "/content/drive/My Drive/PgpTextMining"
OUTPUT_FILE = "pmids.txt"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, OUTPUT_FILE)

# Mount Google Drive
drive.mount('/content/drive')
os.makedirs(OUTPUT_DIR, exist_ok=True)
def get_year_ranges(start, end, interval):
    return [(y, min(y + interval - 1, end)) for y in range(start, end + 1, interval)]
def fetch_pmids_for_range(start_y, end_y):
    query = f'{QUERY_BASE} AND ("{start_y}"[PDAT] : "{end_y}"[PDAT])'
    handle = Entrez.esearch(db="pubmed", term=query, usehistory="y", retmax=0)
    result = Entrez.read(handle)
    handle.close()
    count = int(result["Count"])
    print(f" → Found {count} PMIDs")
    if count > MAX_RECORDS:
        raise RuntimeError(f"Too many results in {start_y}--{end_y} ({count}). Reduce interval.")
    pmids = []
    webenv = result["WebEnv"]
    query_key = result["QueryKey"]
    for start in range(0, count, RETMAX):
        print(f" Fetching {start + 1}--{min(start + RETMAX, count)}")
        handle = Entrez.esearch(
            db="pubmed",
            term=query,
            webenv=webenv,
            query_key=query_key,
            retstart=start,
            retmax=RETMAX,
            retmode="xml"
        )


        data = Entrez.read(handle)
        handle.close()
        pmids.extend(data["IdList"])
        sleep(SLEEP_TIME)

    return pmids

# --- Main Process ---

all_pmids = []
for start_y, end_y in get_year_ranges(START_YEAR, END_YEAR, INTERVAL_YEARS):
    print(f"\nSearching {start_y}--{end_y}")
    all_pmids.extend(fetch_pmids_for_range(start_y, end_y))

# Deduplicate and save
unique_pmids = list(dict.fromkeys(all_pmids))
with open(OUTPUT_PATH, "w") as f:
    f.write("\n".join(unique_pmids))

print(f"\nDone! {len(unique_pmids)} unique PMIDs saved to: {OUTPUT_PATH}")

In [ ]:
# --- 1. Import required libraries ---
from Bio import Entrez
import json
import os
import time
from google.colab import drive

# --- 2. Configuration ---
Entrez.email = "junghosohn8@gmail.com"

# Mount Google Drive
drive.mount('/content/drive')

# Define file paths and chunk size
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
PMID_FILE = os.path.join(BASE_DIR, "pmids.txt")
CHUNK_LIST_FILE = os.path.join(BASE_DIR, "chunk_json_list.txt")
CHUNK_SIZE = 500

# --- 3. Define the Optimized Prompt for ChatGPT ---
prompt_messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": (
        "From the given PubMed abstract, extract all interactions between P-glycoprotein and substances.\n\n"
        "Instructions:\n"
        "- Output a tab-delimited table without column headers. Use a single tab character ('\\t') to separate fields.\n"
        "- Do not wrap the output in code blocks (e.g., triple backticks) or any other formatting.\n"
        "- Output only the raw data rows—no explanations, no headings, no labels.\n"
        "- Each row must represent one unique interaction between P-glycoprotein and a substance.\n"
        "- For each interaction, include the following 4 fields, in this exact order:\n"
        "  1. Transporter name (use \"P-glycoprotein\")\n"
        "  2. Substance name\n"
        "  3. Interaction type (select one of: substrate, inhibitor, inducer, or other)\n"
        "  4. Mechanism of action (if specified; otherwise leave blank)\n"
        "- If the abstract contains no interactions, return only: none\n"
        "- Expand substance abbreviations only if their full form is defined in the abstract.\n\n"
        "Definitions:\n"
        "- Substrate: A substance transported by P-glycoprotein.\n"
        "- Inhibitor: A substance that reduces the transport activity of P-glycoprotein.\n"
        "- Inducer: A substance that increases the expression or activity of P-glycoprotein."
    )},
    {"role": "assistant", "content": "Please provide me with the abstract."}
]

# --- 4. Main Execution Block ---
print(f"Reading PMIDs from: {PMID_FILE}")
with open(PMID_FILE, 'r') as f:
    pmids = [line.strip() for line in f.readlines()]

print(f"Found {len(pmids)} PMIDs. Processing in chunks of {CHUNK_SIZE}...")

chunk_filenames = []
for i in range(0, len(pmids), CHUNK_SIZE):
    chunk_pmids = pmids[i:i + CHUNK_SIZE]
    chunk_filename = f"chunk_{i // CHUNK_SIZE}.jsonl"
    chunk_path = os.path.join(BASE_DIR, chunk_filename)
    chunk_filenames.append(chunk_filename)

    print(f"\nProcessing chunk {i // CHUNK_SIZE + 1} ({len(chunk_pmids)} PMIDs)...")
    print(f"Fetching abstracts...")

    # Fetch abstracts from PubMed using Entrez.efetch
    try:
        handle = Entrez.efetch(db="pubmed", id=chunk_pmids, rettype="medline", retmode="xml")
        records = Entrez.read(handle)
        handle.close()
    except Exception as e:
        print(f"  - An error occurred during fetching: {e}. Skipping this chunk.")
        continue

    print("Formatting data into JSONL file...")
    with open(chunk_path, 'w', encoding='utf-8') as outfile:
        # Check if records were actually returned
        if 'PubmedArticle' in records:
            for record in records['PubmedArticle']:
                pmid = record['MedlineCitation']['PMID']
                article = record['MedlineCitation']['Article']

                if 'Abstract' in article and 'AbstractText' in article['Abstract']:
                    abstract_text = ' '.join(article['Abstract']['AbstractText'])

                    # Create the JSON object for each API request
                    json_request = {
                        "custom_id": f"request-{pmid}",
                        "method": "POST",
                        "url": "/v1/chat/completions",
                        "body": {
                            # Corrected model name for the API call.
                            "model": "o3-mini",
                            "messages": prompt_messages + [{"role": "user", "content": abstract_text}],
                            "max_completion_tokens": 4000,

                        }
                    }
                    outfile.write(json.dumps(json_request) + '\n')
                else:
                    print(f"  - Warning: PMID {pmid} has no abstract. Skipping.")
        else:
            print("  - No valid records returned from Entrez for this chunk.")


    print(f"Chunk successfully saved to: {chunk_path}")
    # A small delay to be polite to the NCBI server between fetching chunks
    time.sleep(1)

# Save the list of all created chunk filenames for the next step
with open(CHUNK_LIST_FILE, 'w') as f:
    for name in chunk_filenames:
        f.write(name + '\n')

print(f"\nAll chunks processed! A list of filenames has been saved to: {CHUNK_LIST_FILE}")

In [ ]:
import os
from google.colab import drive

# Google Drive 마운트
drive.mount('/content/drive')

# 파일 경로 설정
FOLDER_PATH = "/content/drive/My Drive/PgpTextMining"
INPUT_FILE = "chunk_json_list.txt"

# 원본 파일 읽기
input_path = os.path.join(FOLDER_PATH, INPUT_FILE)

with open(input_path, "r", encoding="utf-8") as f:
    all_lines = [line.strip() for line in f if line.strip()]

total_files = len(all_lines)
print(f"총 {total_files}개 파일을 4개 그룹으로 나눕니다.")

# 4개 그룹으로 나누기 (7, 7, 8, 8)
group_sizes = [7, 7, 8, 8]
start_idx = 0

for group_num, group_size in enumerate(group_sizes, 1):
    end_idx = start_idx + group_size
    group_lines = all_lines[start_idx:end_idx]

    # 각 그룹별 파일 생성
    output_file = f"chunk_json_list_{group_num}.txt"
    output_path = os.path.join(FOLDER_PATH, output_file)

    with open(output_path, "w", encoding="utf-8") as f:
        for line in group_lines:
            f.write(line + "\n")

    print(f"그룹 {group_num}: {output_file} ({group_size}개 파일)")
    start_idx = end_idx

print("파일 분할 완료!")

In [ ]:
import os #Listing S2
import time
import json
import requests
from google.colab import drive

# --- Configuration ---
API_KEY = ""
BASE_URL = "https://api.openai.com/v1"
HEADERS = {'Authorization': f'Bearer {API_KEY}'}
DRIVE_PATH = "/content/drive"
FOLDER_PATH = os.path.join(DRIVE_PATH, "My Drive/PgpTextMining")

# --- Separating Groups ---
GROUP_NUMBER = 1

INPUT_FILE = f"chunk_json_list_{GROUP_NUMBER}.txt"
OUTPUT_FILE = f"batch_id_list_{GROUP_NUMBER}.txt"

# --- Initialize session ---
session = requests.Session()
session.headers.update(HEADERS)

# --- Utility functions ---
def request_with_retry(method, url, **kwargs):
    """Send HTTP request with retry logic."""
    for attempt in range(3):
        try:
            response = session.request(method, url, **kwargs)
            if response.ok:
                return response.json()
            print(f"[HTTP {response.status_code}] {response.text}")
        except Exception as e:
            print(f"Request error (attempt {attempt + 1}): {e}")
        time.sleep(2)
    return None

def upload_file(filepath, purpose="batch"):
    """Upload a file to OpenAI for batch processing."""
    if not os.path.exists(filepath):
        print(f"File not found: {filepath}")
        return None

    with open(filepath, 'rb') as f:
        files = {'file': f, 'purpose': (None, purpose)}
        return request_with_retry("POST", f"{BASE_URL}/files", files=files)

def start_batch(file_id):
    """Start batch processing with the given file ID."""
    data = {
        'input_file_id': file_id,
        'endpoint': '/v1/chat/completions',
        'completion_window': '24h'
    }
    res = request_with_retry("POST", f"{BASE_URL}/batches", json=data)
    return res.get("id") if res else None

def process_files():
    """Upload input files and start batch jobs. Save batch IDs."""
    input_path = os.path.join(FOLDER_PATH, INPUT_FILE)
    output_path = os.path.join(FOLDER_PATH, OUTPUT_FILE)

    if not os.path.exists(input_path):
        print(f"Input file not found: {input_path}")
        return

    with open(input_path, "r", encoding="utf-8") as f:
        filenames = [line.strip() for line in f if line.strip()]

    if not filenames:
        print("No filenames found in input.")
        return

    with open(output_path, "a", encoding="utf-8") as out_f:
        for name in filenames:
            filepath = os.path.join(FOLDER_PATH, name)
            file_res = upload_file(filepath)

            if file_res and (file_id := file_res.get("id")):
                batch_id = start_batch(file_id)
                if batch_id:
                    print(f"Uploaded: {name}, Batch ID: {batch_id}")
                    out_f.write(batch_id + "\n")
                else:
                    print(f"Failed to start batch for: {name}")
            else:
                print(f"Failed to upload file: {name}")

            time.sleep(180)

# --- Execute ---
if __name__ == "__main__":
    drive.mount(DRIVE_PATH, force_remount=True)
    os.makedirs(FOLDER_PATH, exist_ok=True)
    process_files()

In [ ]:
import os #Listing S3
import json
import time
import requests
from google.colab import drive

# --- Configuration ---
API_KEY = ""
BASE_URL = "https://api.openai.com/v1"
HEADERS = {'Authorization': f'Bearer {API_KEY}'}
DRIVE_PATH = "/content/drive"
FOLDER_PATH = os.path.join(DRIVE_PATH, "My Drive/PgpTextMining")

# --- Group number ---
GROUP_NUMBER = 1

INPUT_FILE = f"batch_id_list_{GROUP_NUMBER}.txt"
OUTPUT_FILE = f"result_{GROUP_NUMBER}.txt"

# --- Initialize session ---
session = requests.Session()
session.headers.update(HEADERS)

# --- Utility functions ---
def request_with_retry(method, url, **kwargs):
    """Send HTTP request with retry logic."""
    for attempt in range(3):
        try:
            response = session.request(method, url, **kwargs)
            if response.ok:
                return response.json()
            print(f"[HTTP {response.status_code}] {response.text}")
        except Exception as e:
            print(f"Request error (attempt {attempt + 1}): {e}")
        time.sleep(2)
    return None

def check_batch_status(batch_id):
    """Return (status, output_file_id) for a given batch ID."""
    res = request_with_retry("GET", f"{BASE_URL}/batches/{batch_id}")
    if not res:
        print(f"Failed to retrieve status for batch: {batch_id}")
        return None, None
    return res.get("status"), res.get("output_file_id")

def download_results(file_id):
    """Download output file content from OpenAI."""
    try:
        response = session.get(f"{BASE_URL}/files/{file_id}/content")
        if response.ok:
            return response.text
        print(f"Download failed for file {file_id} (status {response.status_code})")
    except Exception as e:
        print(f"Download error for file {file_id}: {e}")
    return None

def parse_and_write_results(response_text, output_file):
    """Parse OpenAI batch response and write formatted result."""
    for line in response_text.splitlines():
        time.sleep(0.02)  # Throttle for readability or rate-limiting
        try:
            json_obj = json.loads(line)
            pmid = json_obj.get("custom_id", "unknown").split("-")[1] if "custom_id" in json_obj else "unknown"
            choices = json_obj.get("response", {}).get("body", {}).get("choices", [])
            if choices:
                content = choices[0].get("message", {}).get("content", "")
                for entry in content.splitlines():
                    clean_entry = entry.replace('\\t', '\t')
                    output_file.write(f"{pmid}\t{clean_entry}\n")
        except (json.JSONDecodeError, IndexError) as e:
            print(f"Error parsing line: {e}")

# --- Main process ---
def main():
    drive.mount(DRIVE_PATH, force_remount=True)
    os.makedirs(FOLDER_PATH, exist_ok=True)

    input_path = os.path.join(FOLDER_PATH, INPUT_FILE)
    output_path = os.path.join(FOLDER_PATH, OUTPUT_FILE)

    if not os.path.exists(input_path):
        print(f"Input file not found: {input_path}")
        return

    with open(input_path, "r", encoding="utf-8") as f:
        batch_ids = [line.strip() for line in f if line.strip()]

    output_ids = []
    for batch_id in batch_ids:
        status, file_id = check_batch_status(batch_id)
        if status == "completed":
            output_ids.append(file_id)
        elif status == "in_progress":
            print(f"Batch {batch_id} is still in progress.")
            return
        else:
            print(f"Batch {batch_id} has failed or expired.")

    if not output_ids:
        print("No completed batch outputs found.")
        return

    with open(output_path, "w", encoding="utf-8") as out_file:
        for file_id in output_ids:
            result_text = download_results(file_id)
            if result_text:
                parse_and_write_results(result_text, out_file)

    print(f"Results written to: {output_path}")

# --- Execute ---
if __name__ == "__main__":
    main()

In [ ]:
import os #Listing S2
import time
import json
import requests
from google.colab import drive

# --- Configuration ---
API_KEY = ""
BASE_URL = "https://api.openai.com/v1"
HEADERS = {'Authorization': f'Bearer {API_KEY}'}
DRIVE_PATH = "/content/drive"
FOLDER_PATH = os.path.join(DRIVE_PATH, "My Drive/PgpTextMining")

# --- Separating Groups ---
GROUP_NUMBER = 2

INPUT_FILE = f"chunk_json_list_{GROUP_NUMBER}.txt"
OUTPUT_FILE = f"batch_id_list_{GROUP_NUMBER}.txt"

# --- Initialize session ---
session = requests.Session()
session.headers.update(HEADERS)

# --- Utility functions ---
def request_with_retry(method, url, **kwargs):
    """Send HTTP request with retry logic."""
    for attempt in range(3):
        try:
            response = session.request(method, url, **kwargs)
            if response.ok:
                return response.json()
            print(f"[HTTP {response.status_code}] {response.text}")
        except Exception as e:
            print(f"Request error (attempt {attempt + 1}): {e}")
        time.sleep(2)
    return None

def upload_file(filepath, purpose="batch"):
    """Upload a file to OpenAI for batch processing."""
    if not os.path.exists(filepath):
        print(f"File not found: {filepath}")
        return None

    with open(filepath, 'rb') as f:
        files = {'file': f, 'purpose': (None, purpose)}
        return request_with_retry("POST", f"{BASE_URL}/files", files=files)

def start_batch(file_id):
    """Start batch processing with the given file ID."""
    data = {
        'input_file_id': file_id,
        'endpoint': '/v1/chat/completions',
        'completion_window': '24h'
    }
    res = request_with_retry("POST", f"{BASE_URL}/batches", json=data)
    return res.get("id") if res else None

def process_files():
    """Upload input files and start batch jobs. Save batch IDs."""
    input_path = os.path.join(FOLDER_PATH, INPUT_FILE)
    output_path = os.path.join(FOLDER_PATH, OUTPUT_FILE)

    if not os.path.exists(input_path):
        print(f"Input file not found: {input_path}")
        return

    with open(input_path, "r", encoding="utf-8") as f:
        filenames = [line.strip() for line in f if line.strip()]

    if not filenames:
        print("No filenames found in input.")
        return

    with open(output_path, "a", encoding="utf-8") as out_f:
        for name in filenames:
            filepath = os.path.join(FOLDER_PATH, name)
            file_res = upload_file(filepath)

            if file_res and (file_id := file_res.get("id")):
                batch_id = start_batch(file_id)
                if batch_id:
                    print(f"Uploaded: {name}, Batch ID: {batch_id}")
                    out_f.write(batch_id + "\n")
                else:
                    print(f"Failed to start batch for: {name}")
            else:
                print(f"Failed to upload file: {name}")

            time.sleep(180)

# --- Execute ---
if __name__ == "__main__":
    drive.mount(DRIVE_PATH, force_remount=True)
    os.makedirs(FOLDER_PATH, exist_ok=True)
    process_files()

In [ ]:
import os #Listing S3
import json
import time
import requests
from google.colab import drive

# --- Configuration ---
API_KEY = ""
BASE_URL = "https://api.openai.com/v1"
HEADERS = {'Authorization': f'Bearer {API_KEY}'}
DRIVE_PATH = "/content/drive"
FOLDER_PATH = os.path.join(DRIVE_PATH, "My Drive/PgpTextMining")

# --- Group number ---
GROUP_NUMBER = 2

INPUT_FILE = f"batch_id_list_{GROUP_NUMBER}.txt"
OUTPUT_FILE = f"result_{GROUP_NUMBER}.txt"

# --- Initialize session ---
session = requests.Session()
session.headers.update(HEADERS)

# --- Utility functions ---
def request_with_retry(method, url, **kwargs):
    """Send HTTP request with retry logic."""
    for attempt in range(3):
        try:
            response = session.request(method, url, **kwargs)
            if response.ok:
                return response.json()
            print(f"[HTTP {response.status_code}] {response.text}")
        except Exception as e:
            print(f"Request error (attempt {attempt + 1}): {e}")
        time.sleep(2)
    return None

def check_batch_status(batch_id):
    """Return (status, output_file_id) for a given batch ID."""
    res = request_with_retry("GET", f"{BASE_URL}/batches/{batch_id}")
    if not res:
        print(f"Failed to retrieve status for batch: {batch_id}")
        return None, None
    return res.get("status"), res.get("output_file_id")

def download_results(file_id):
    """Download output file content from OpenAI."""
    try:
        response = session.get(f"{BASE_URL}/files/{file_id}/content")
        if response.ok:
            return response.text
        print(f"Download failed for file {file_id} (status {response.status_code})")
    except Exception as e:
        print(f"Download error for file {file_id}: {e}")
    return None

def parse_and_write_results(response_text, output_file):
    """Parse OpenAI batch response and write formatted result."""
    for line in response_text.splitlines():
        time.sleep(0.02)  # Throttle for readability or rate-limiting
        try:
            json_obj = json.loads(line)
            pmid = json_obj.get("custom_id", "unknown").split("-")[1] if "custom_id" in json_obj else "unknown"
            choices = json_obj.get("response", {}).get("body", {}).get("choices", [])
            if choices:
                content = choices[0].get("message", {}).get("content", "")
                for entry in content.splitlines():
                    clean_entry = entry.replace('\\t', '\t')
                    output_file.write(f"{pmid}\t{clean_entry}\n")
        except (json.JSONDecodeError, IndexError) as e:
            print(f"Error parsing line: {e}")

# --- Main process ---
def main():
    drive.mount(DRIVE_PATH, force_remount=True)
    os.makedirs(FOLDER_PATH, exist_ok=True)

    input_path = os.path.join(FOLDER_PATH, INPUT_FILE)
    output_path = os.path.join(FOLDER_PATH, OUTPUT_FILE)

    if not os.path.exists(input_path):
        print(f"Input file not found: {input_path}")
        return

    with open(input_path, "r", encoding="utf-8") as f:
        batch_ids = [line.strip() for line in f if line.strip()]

    output_ids = []
    for batch_id in batch_ids:
        status, file_id = check_batch_status(batch_id)
        if status == "completed":
            output_ids.append(file_id)
        elif status == "in_progress":
            print(f"Batch {batch_id} is still in progress.")
            return
        else:
            print(f"Batch {batch_id} has failed or expired.")

    if not output_ids:
        print("No completed batch outputs found.")
        return

    with open(output_path, "w", encoding="utf-8") as out_file:
        for file_id in output_ids:
            result_text = download_results(file_id)
            if result_text:
                parse_and_write_results(result_text, out_file)

    print(f"Results written to: {output_path}")

# --- Execute ---
if __name__ == "__main__":
    main()

In [ ]:
import os #Listing S2
import time
import json
import requests
from google.colab import drive

# --- Configuration ---
API_KEY = ""
BASE_URL = "https://api.openai.com/v1"
HEADERS = {'Authorization': f'Bearer {API_KEY}'}
DRIVE_PATH = "/content/drive"
FOLDER_PATH = os.path.join(DRIVE_PATH, "My Drive/PgpTextMining")

# --- Separating Groups ---
GROUP_NUMBER = 3

INPUT_FILE = f"chunk_json_list_{GROUP_NUMBER}.txt"
OUTPUT_FILE = f"batch_id_list_{GROUP_NUMBER}.txt"

# --- Initialize session ---
session = requests.Session()
session.headers.update(HEADERS)

# --- Utility functions ---
def request_with_retry(method, url, **kwargs):
    """Send HTTP request with retry logic."""
    for attempt in range(3):
        try:
            response = session.request(method, url, **kwargs)
            if response.ok:
                return response.json()
            print(f"[HTTP {response.status_code}] {response.text}")
        except Exception as e:
            print(f"Request error (attempt {attempt + 1}): {e}")
        time.sleep(2)
    return None

def upload_file(filepath, purpose="batch"):
    """Upload a file to OpenAI for batch processing."""
    if not os.path.exists(filepath):
        print(f"File not found: {filepath}")
        return None

    with open(filepath, 'rb') as f:
        files = {'file': f, 'purpose': (None, purpose)}
        return request_with_retry("POST", f"{BASE_URL}/files", files=files)

def start_batch(file_id):
    """Start batch processing with the given file ID."""
    data = {
        'input_file_id': file_id,
        'endpoint': '/v1/chat/completions',
        'completion_window': '24h'
    }
    res = request_with_retry("POST", f"{BASE_URL}/batches", json=data)
    return res.get("id") if res else None

def process_files():
    """Upload input files and start batch jobs. Save batch IDs."""
    input_path = os.path.join(FOLDER_PATH, INPUT_FILE)
    output_path = os.path.join(FOLDER_PATH, OUTPUT_FILE)

    if not os.path.exists(input_path):
        print(f"Input file not found: {input_path}")
        return

    with open(input_path, "r", encoding="utf-8") as f:
        filenames = [line.strip() for line in f if line.strip()]

    if not filenames:
        print("No filenames found in input.")
        return

    with open(output_path, "a", encoding="utf-8") as out_f:
        for name in filenames:
            filepath = os.path.join(FOLDER_PATH, name)
            file_res = upload_file(filepath)

            if file_res and (file_id := file_res.get("id")):
                batch_id = start_batch(file_id)
                if batch_id:
                    print(f"Uploaded: {name}, Batch ID: {batch_id}")
                    out_f.write(batch_id + "\n")
                else:
                    print(f"Failed to start batch for: {name}")
            else:
                print(f"Failed to upload file: {name}")

            time.sleep(180)

# --- Execute ---
if __name__ == "__main__":
    drive.mount(DRIVE_PATH, force_remount=True)
    os.makedirs(FOLDER_PATH, exist_ok=True)
    process_files()

In [ ]:
import os #Listing S3
import json
import time
import requests
from google.colab import drive

# --- Configuration ---
API_KEY = ""
BASE_URL = "https://api.openai.com/v1"
HEADERS = {'Authorization': f'Bearer {API_KEY}'}
DRIVE_PATH = "/content/drive"
FOLDER_PATH = os.path.join(DRIVE_PATH, "My Drive/PgpTextMining")

# --- Group number ---
GROUP_NUMBER = 3

INPUT_FILE = f"batch_id_list_{GROUP_NUMBER}.txt"
OUTPUT_FILE = f"result_{GROUP_NUMBER}.txt"

# --- Initialize session ---
session = requests.Session()
session.headers.update(HEADERS)

# --- Utility functions ---
def request_with_retry(method, url, **kwargs):
    """Send HTTP request with retry logic."""
    for attempt in range(3):
        try:
            response = session.request(method, url, **kwargs)
            if response.ok:
                return response.json()
            print(f"[HTTP {response.status_code}] {response.text}")
        except Exception as e:
            print(f"Request error (attempt {attempt + 1}): {e}")
        time.sleep(2)
    return None

def check_batch_status(batch_id):
    """Return (status, output_file_id) for a given batch ID."""
    res = request_with_retry("GET", f"{BASE_URL}/batches/{batch_id}")
    if not res:
        print(f"Failed to retrieve status for batch: {batch_id}")
        return None, None
    return res.get("status"), res.get("output_file_id")

def download_results(file_id):
    """Download output file content from OpenAI."""
    try:
        response = session.get(f"{BASE_URL}/files/{file_id}/content")
        if response.ok:
            return response.text
        print(f"Download failed for file {file_id} (status {response.status_code})")
    except Exception as e:
        print(f"Download error for file {file_id}: {e}")
    return None

def parse_and_write_results(response_text, output_file):
    """Parse OpenAI batch response and write formatted result."""
    for line in response_text.splitlines():
        time.sleep(0.02)  # Throttle for readability or rate-limiting
        try:
            json_obj = json.loads(line)
            pmid = json_obj.get("custom_id", "unknown").split("-")[1] if "custom_id" in json_obj else "unknown"
            choices = json_obj.get("response", {}).get("body", {}).get("choices", [])
            if choices:
                content = choices[0].get("message", {}).get("content", "")
                for entry in content.splitlines():
                    clean_entry = entry.replace('\\t', '\t')
                    output_file.write(f"{pmid}\t{clean_entry}\n")
        except (json.JSONDecodeError, IndexError) as e:
            print(f"Error parsing line: {e}")

# --- Main process ---
def main():
    drive.mount(DRIVE_PATH, force_remount=True)
    os.makedirs(FOLDER_PATH, exist_ok=True)

    input_path = os.path.join(FOLDER_PATH, INPUT_FILE)
    output_path = os.path.join(FOLDER_PATH, OUTPUT_FILE)

    if not os.path.exists(input_path):
        print(f"Input file not found: {input_path}")
        return

    with open(input_path, "r", encoding="utf-8") as f:
        batch_ids = [line.strip() for line in f if line.strip()]

    output_ids = []
    for batch_id in batch_ids:
        status, file_id = check_batch_status(batch_id)
        if status == "completed":
            output_ids.append(file_id)
        elif status == "in_progress":
            print(f"Batch {batch_id} is still in progress.")
            return
        else:
            print(f"Batch {batch_id} has failed or expired.")

    if not output_ids:
        print("No completed batch outputs found.")
        return

    with open(output_path, "w", encoding="utf-8") as out_file:
        for file_id in output_ids:
            result_text = download_results(file_id)
            if result_text:
                parse_and_write_results(result_text, out_file)

    print(f"Results written to: {output_path}")

# --- Execute ---
if __name__ == "__main__":
    main()

In [ ]:
import os #Listing S2
import time
import json
import requests
from google.colab import drive

# --- Configuration ---
API_KEY = ""
BASE_URL = "https://api.openai.com/v1"
HEADERS = {'Authorization': f'Bearer {API_KEY}'}
DRIVE_PATH = "/content/drive"
FOLDER_PATH = os.path.join(DRIVE_PATH, "My Drive/PgpTextMining")

# --- Separating Groups ---
GROUP_NUMBER = 4

INPUT_FILE = f"chunk_json_list_{GROUP_NUMBER}.txt"
OUTPUT_FILE = f"batch_id_list_{GROUP_NUMBER}.txt"

# --- Initialize session ---
session = requests.Session()
session.headers.update(HEADERS)

# --- Utility functions ---
def request_with_retry(method, url, **kwargs):
    """Send HTTP request with retry logic."""
    for attempt in range(3):
        try:
            response = session.request(method, url, **kwargs)
            if response.ok:
                return response.json()
            print(f"[HTTP {response.status_code}] {response.text}")
        except Exception as e:
            print(f"Request error (attempt {attempt + 1}): {e}")
        time.sleep(2)
    return None

def upload_file(filepath, purpose="batch"):
    """Upload a file to OpenAI for batch processing."""
    if not os.path.exists(filepath):
        print(f"File not found: {filepath}")
        return None

    with open(filepath, 'rb') as f:
        files = {'file': f, 'purpose': (None, purpose)}
        return request_with_retry("POST", f"{BASE_URL}/files", files=files)

def start_batch(file_id):
    """Start batch processing with the given file ID."""
    data = {
        'input_file_id': file_id,
        'endpoint': '/v1/chat/completions',
        'completion_window': '24h'
    }
    res = request_with_retry("POST", f"{BASE_URL}/batches", json=data)
    return res.get("id") if res else None

def process_files():
    """Upload input files and start batch jobs. Save batch IDs."""
    input_path = os.path.join(FOLDER_PATH, INPUT_FILE)
    output_path = os.path.join(FOLDER_PATH, OUTPUT_FILE)

    if not os.path.exists(input_path):
        print(f"Input file not found: {input_path}")
        return

    with open(input_path, "r", encoding="utf-8") as f:
        filenames = [line.strip() for line in f if line.strip()]

    if not filenames:
        print("No filenames found in input.")
        return

    with open(output_path, "a", encoding="utf-8") as out_f:
        for name in filenames:
            filepath = os.path.join(FOLDER_PATH, name)
            file_res = upload_file(filepath)

            if file_res and (file_id := file_res.get("id")):
                batch_id = start_batch(file_id)
                if batch_id:
                    print(f"Uploaded: {name}, Batch ID: {batch_id}")
                    out_f.write(batch_id + "\n")
                else:
                    print(f"Failed to start batch for: {name}")
            else:
                print(f"Failed to upload file: {name}")

            time.sleep(180)

# --- Execute ---
if __name__ == "__main__":
    drive.mount(DRIVE_PATH, force_remount=True)
    os.makedirs(FOLDER_PATH, exist_ok=True)
    process_files()

In [ ]:
import os #Listing S3
import json
import time
import requests
from google.colab import drive

# --- Configuration ---
API_KEY = ""
BASE_URL = "https://api.openai.com/v1"
HEADERS = {'Authorization': f'Bearer {API_KEY}'}
DRIVE_PATH = "/content/drive"
FOLDER_PATH = os.path.join(DRIVE_PATH, "My Drive/PgpTextMining")

# --- Group number ---
GROUP_NUMBER = 4

INPUT_FILE = f"batch_id_list_{GROUP_NUMBER}.txt"
OUTPUT_FILE = f"result_{GROUP_NUMBER}.txt"

# --- Initialize session ---
session = requests.Session()
session.headers.update(HEADERS)

# --- Utility functions ---
def request_with_retry(method, url, **kwargs):
    """Send HTTP request with retry logic."""
    for attempt in range(3):
        try:
            response = session.request(method, url, **kwargs)
            if response.ok:
                return response.json()
            print(f"[HTTP {response.status_code}] {response.text}")
        except Exception as e:
            print(f"Request error (attempt {attempt + 1}): {e}")
        time.sleep(2)
    return None

def check_batch_status(batch_id):
    """Return (status, output_file_id) for a given batch ID."""
    res = request_with_retry("GET", f"{BASE_URL}/batches/{batch_id}")
    if not res:
        print(f"Failed to retrieve status for batch: {batch_id}")
        return None, None
    return res.get("status"), res.get("output_file_id")

def download_results(file_id):
    """Download output file content from OpenAI."""
    try:
        response = session.get(f"{BASE_URL}/files/{file_id}/content")
        if response.ok:
            return response.text
        print(f"Download failed for file {file_id} (status {response.status_code})")
    except Exception as e:
        print(f"Download error for file {file_id}: {e}")
    return None

def parse_and_write_results(response_text, output_file):
    """Parse OpenAI batch response and write formatted result."""
    for line in response_text.splitlines():
        time.sleep(0.02)  # Throttle for readability or rate-limiting
        try:
            json_obj = json.loads(line)
            pmid = json_obj.get("custom_id", "unknown").split("-")[1] if "custom_id" in json_obj else "unknown"
            choices = json_obj.get("response", {}).get("body", {}).get("choices", [])
            if choices:
                content = choices[0].get("message", {}).get("content", "")
                for entry in content.splitlines():
                    clean_entry = entry.replace('\\t', '\t')
                    output_file.write(f"{pmid}\t{clean_entry}\n")
        except (json.JSONDecodeError, IndexError) as e:
            print(f"Error parsing line: {e}")

# --- Main process ---
def main():
    drive.mount(DRIVE_PATH, force_remount=True)
    os.makedirs(FOLDER_PATH, exist_ok=True)

    input_path = os.path.join(FOLDER_PATH, INPUT_FILE)
    output_path = os.path.join(FOLDER_PATH, OUTPUT_FILE)

    if not os.path.exists(input_path):
        print(f"Input file not found: {input_path}")
        return

    with open(input_path, "r", encoding="utf-8") as f:
        batch_ids = [line.strip() for line in f if line.strip()]

    output_ids = []
    for batch_id in batch_ids:
        status, file_id = check_batch_status(batch_id)
        if status == "completed":
            output_ids.append(file_id)
        elif status == "in_progress":
            print(f"Batch {batch_id} is still in progress.")
            return
        else:
            print(f"Batch {batch_id} has failed or expired.")

    if not output_ids:
        print("No completed batch outputs found.")
        return

    with open(output_path, "w", encoding="utf-8") as out_file:
        for file_id in output_ids:
            result_text = download_results(file_id)
            if result_text:
                parse_and_write_results(result_text, out_file)

    print(f"Results written to: {output_path}")

# --- Execute ---
if __name__ == "__main__":
    main()

In [ ]:
import os
import requests
import time
from google.colab import drive

# --- Configuration ---
# Please replace with your actual API key
API_KEY = ""
BASE_URL = "https://api.openai.com/v1"
HEADERS = {'Authorization': f'Bearer {API_KEY}'}

# Mount Google Drive
drive.mount('/content/drive')
BASE_DIR = "/content/drive/My Drive/PgpTextMining"

# --- Function to check batch status ---
def get_failed_request_count(batch_id):
    """
    Fetches the status for a single batch ID and returns the number of failed requests.
    This function isolates the logic for one API call, making the main code cleaner.
    """
    try:
        response = requests.get(f"{BASE_URL}/batches/{batch_id}", headers=HEADERS)
        if response.ok:
            status_info = response.json()
            # This is the core logic for this function:
            # It safely gets the number of failed requests from the JSON response.
            # .get() is used to avoid errors if a key doesn't exist.
            failed_count = status_info.get('request_counts', {}).get('failed', 0)
            return failed_count
        else:
            print(f"  - Error checking {batch_id}: HTTP {response.status_code}")
            return 0 # Return 0 if we can't get the status
    except requests.exceptions.RequestException as e:
        print(f"  - Network error checking {batch_id}: {e}")
        return 0

# --- Main process ---
# This loop will run 4 times, for group 1, 2, 3, and 4.
for group_num in range(1, 5):
    batch_list_file = os.path.join(BASE_DIR, f"batch_id_list_{group_num}.txt")
    total_failed_for_group = 0

    if not os.path.exists(batch_list_file):
        print(f"\nFile not found for Group {group_num}: {batch_list_file}")
        continue

    print(f"\n--- Processing Group {group_num} ---")
    with open(batch_list_file, 'r') as f:
        # Reads all batch IDs from the current group file.
        batch_ids = [line.strip() for line in f if line.strip()]

    print(f"Found {len(batch_ids)} batches to check.")

    # This loop iterates through each batch ID within the group.
    for batch_id in batch_ids:
        failed_count = get_failed_request_count(batch_id)
        total_failed_for_group += failed_count
        # This print statement shows the progress for each batch ID.
        print(f"  - Batch {batch_id}: {failed_count} failed requests.")
        # A short delay to be polite to the OpenAI API and avoid rate limits.
        time.sleep(1)

    # After checking all batches in the group, print the summary.
    # This gives us the final answer for each batch group file.
    print(f"\n>>> Summary for Group {group_num}: Total Failed Requests = {total_failed_for_group}")

print("\nAll groups checked.")

In [ ]:
import os
import requests
import json
from google.colab import drive

# --- ⚙️ 설정 (Configuration) ---
# 1. 연구원님의 API 키를 입력해주세요.
API_KEY = ""

# 2. 실패가 발생한 Batch ID를 입력해주세요.
BATCH_ID = "batch_686f43e4b1a881908f85bc793b9614a1"

# 3. ‼️ 가장 중요: 위 Batch ID를 만들 때 사용된 '원본 요청서' 파일 이름을 정확히 입력해주세요.
INPUT_CHUNK_FILE = "chunk_18.jsonl"

# --- 경로 설정 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
INPUT_FILE_PATH = os.path.join(BASE_DIR, INPUT_CHUNK_FILE)
HEADERS = {'Authorization': f'Bearer {API_KEY}'}
BASE_URL = "https://api.openai.com/v1"

# --- 함수 정의 (Functions) ---
def get_batch_info(batch_id):
    """Batch의 상태와 결과 파일 ID를 가져옵니다."""
    print(f"1. Batch 상태 확인 중: {batch_id}")
    response = requests.get(f"{BASE_URL}/batches/{batch_id}", headers=HEADERS)
    if response.ok:
        return response.json()
    return None

def get_pmids_from_api_result(file_id):
    """OpenAI 결과 파일에서 성공한 PMID 목록을 추출합니다."""
    print(f"2. 성공한 결과 다운로드 중: {file_id}")
    response = requests.get(f"{BASE_URL}/files/{file_id}/content", headers=HEADERS)
    if not response.ok:
        print("   -> 결과 파일 다운로드 실패.")
        return set()

    # 이 set에 성공적으로 처리된 PMID들이 저장됩니다.
    successful_pmids = set()
    for line in response.text.strip().split('\n'):
        try:
            # 각 줄은 JSON 객체입니다.
            result_json = json.loads(line)
            # 결과 JSON 안의 'custom_id'에서 PMID를 추출합니다.
            # custom_id 형식: "request-21538355"
            custom_id = result_json.get('custom_id', '')
            pmid = custom_id.split('-')[1].strip()
            if pmid:
                successful_pmids.add(pmid)
        except (json.JSONDecodeError, IndexError):
            # 줄이 깨졌거나 형식이 다르면 건너뜁니다.
            continue
    return successful_pmids

def get_pmids_from_original_file(input_filepath):
    """우리가 보냈던 원본 파일에서 모든 PMID 목록을 추출합니다."""
    if not os.path.exists(input_filepath):
        print(f"   -> 원본 파일({input_filepath})을 찾을 수 없습니다!")
        return set()

    print(f"3. 원본 요청서에서 PMID 목록 읽는 중: {input_filepath}")
    original_pmids = set()
    with open(input_filepath, 'r') as f:
        for line in f:
            try:
                # 각 줄은 JSON 객체입니다.
                request_json = json.loads(line)
                # 요청 JSON 안의 'custom_id'에서 PMID를 추출합니다.
                custom_id = request_json.get('custom_id', '')
                pmid = custom_id.split('-')[1].strip()
                if pmid:
                    original_pmids.add(pmid)
            except (json.JSONDecodeError, IndexError):
                continue
    return original_pmids

# --- 🕵️ 메인 실행 로직 (Main Execution) ---
batch_info = get_batch_info(BATCH_ID)

if batch_info and batch_info.get('status') == 'completed':
    output_file_id = batch_info.get('output_file_id')

    # 1. 성공한 PMID 목록 가져오기 (499개 예상)
    successful_pmids = get_pmids_from_api_result(output_file_id)
    print(f"   -> {len(successful_pmids)}개의 성공한 PMID를 확인했습니다.")

    # 2. 원본 PMID 목록 가져오기 (500개 예상)
    original_pmids = get_pmids_from_original_file(INPUT_FILE_PATH)
    print(f"   -> {len(original_pmids)}개의 원본 PMID를 확인했습니다.")

    # 3. 두 목록을 비교하여 빠진 것 찾기 (차집합 연산)
    if original_pmids:
        failed_pmids = original_pmids - successful_pmids

        print("\n---  최종 분석 결과 ---")
        if failed_pmids:
            print(f"✅ 다음 {len(failed_pmids)}개의 PMID가 처리 중 실패했습니다:")
            for pmid in failed_pmids:
                print(f"    - 실패한 PMID: {pmid}")
                print(f"    - PubMed 링크: https://pubmed.ncbi.nlm.nih.gov/{pmid}/")
        else:
            print("✅ 모든 PMID가 성공적으로 처리되었습니다. 실패한 항목이 없습니다.")
else:
    print("\n오류: Batch ID를 찾을 수 없거나 아직 작업이 완료되지 않았습니다.")

In [ ]:
import pandas as pd
import os
from collections import Counter, defaultdict
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Configuration
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
RESULT_FILES = [os.path.join(BASE_DIR, f"result_{i}.txt") for i in range(1, 5)]
MERGED_FILE = os.path.join(BASE_DIR, "merged_results.txt")
CHEMICALS_LIST_FILE = os.path.join(BASE_DIR, "chemicals_list.txt")

print("--- Step 1: Merging result files ---")

all_lines = []
total_interactions = 0
none_count = 0

for i, file_path in enumerate(RESULT_FILES, 1):
    if os.path.exists(file_path):
        print(f"Processing result_{i}.txt...")

        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        valid_lines = []
        file_none_count = 0
        file_interactions = 0

        for line in lines:
            line = line.strip()
            if line:  # 빈 줄이 아닌 경우
                parts = line.split('\t')

                # "none"인 경우 체크
                if len(parts) >= 2 and parts[1].lower().strip() == 'none':
                    file_none_count += 1
                    none_count += 1
                    continue

                # 유효한 interaction인지 체크 (최소 4개 컬럼 필요)
                if len(parts) >= 4 and parts[1].strip() and parts[2].strip() and parts[3].strip():
                    # P-glycoprotein 관련 interaction만 추출
                    if 'p-glycoprotein' in parts[1].lower() or 'p-gp' in parts[1].lower() or 'abcb1' in parts[1].lower():
                        valid_lines.append(line)
                        file_interactions += 1
                        total_interactions += 1

        all_lines.extend(valid_lines)
        print(f"  - Valid interactions: {file_interactions}")
        print(f"  - 'None' entries: {file_none_count}")
    else:
        print(f"File not found: {file_path}")

print(f"\n--- Step 2: Writing merged file ---")
print(f"Total valid interactions: {total_interactions}")
print(f"Total 'none' entries: {none_count}")

# 병합된 파일 저장
with open(MERGED_FILE, 'w', encoding='utf-8') as f:
    for line in all_lines:
        f.write(line + '\n')

print(f"Merged file saved: {MERGED_FILE}")

print(f"\n--- Step 3: Creating detailed chemicals list ---")

# 물질별 interaction type 카운트를 저장할 딕셔너리
substance_data = defaultdict(lambda: {'substrate': 0, 'inhibitor': 0, 'inducer': 0, 'other': 0})

for line in all_lines:
    parts = line.split('\t')
    if len(parts) >= 4:
        pmid = parts[0].strip()
        transporter = parts[1].strip()
        substance = parts[2].strip()
        interaction_type = parts[3].strip().lower()

        # Substance 이름 정리 (빈 값, none 제외)
        if substance and substance.lower() not in ['none', 'unknown', 'nan', '']:
            # Interaction type 정규화 및 카운트
            if interaction_type in ['substrate', 'substrates']:
                substance_data[substance]['substrate'] += 1
            elif interaction_type in ['inhibitor', 'inhibitors']:
                substance_data[substance]['inhibitor'] += 1
            elif interaction_type in ['inducer', 'inducers']:
                substance_data[substance]['inducer'] += 1
            else:
                substance_data[substance]['other'] += 1

# 총 언급 횟수 계산 및 정렬
substance_totals = []
for substance, counts in substance_data.items():
    total = counts['substrate'] + counts['inhibitor'] + counts['inducer'] + counts['other']
    substance_totals.append((substance, total, counts))

# 총 언급 횟수로 내림차순 정렬
substance_totals.sort(key=lambda x: x[1], reverse=True)

# chemicals_list.txt 생성 (상세 정보 포함)
with open(CHEMICALS_LIST_FILE, 'w', encoding='utf-8') as f:
    # 헤더 작성 (주석으로)
    f.write("# substance_name\ttotal_count\tsubstrate_count\tinhibitor_count\tinducer_count\tother_count\n")

    for substance, total, counts in substance_totals:
        f.write(f"{substance}\t{total}\t{counts['substrate']}\t{counts['inhibitor']}\t{counts['inducer']}\t{counts['other']}\n")

print(f"Detailed chemicals list saved: {CHEMICALS_LIST_FILE}")

# 통계 요약
print(f"\n--- Step 4: Summary Statistics ---")
print(f"Total unique substances: {len(substance_data)}")

substrate_count = sum(1 for _, _, counts in substance_totals if counts['substrate'] > 0)
inhibitor_count = sum(1 for _, _, counts in substance_totals if counts['inhibitor'] > 0)
inducer_count = sum(1 for _, _, counts in substance_totals if counts['inducer'] > 0)
other_count = sum(1 for _, _, counts in substance_totals if counts['other'] > 0)

print(f"Substances mentioned as substrates: {substrate_count}")
print(f"Substances mentioned as inhibitors: {inhibitor_count}")
print(f"Substances mentioned as inducers: {inducer_count}")
print(f"Substances mentioned as other: {other_count}")

print(f"\n--- Step 5: Top 10 most mentioned substances ---")
print("Rank\tSubstance\tTotal\tSubstrate\tInhibitor\tInducer\tOther")
print("-" * 80)
for i, (substance, total, counts) in enumerate(substance_totals[:10], 1):
    print(f"{i}\t{substance[:20]}\t{total}\t{counts['substrate']}\t{counts['inhibitor']}\t{counts['inducer']}\t{counts['other']}")

print(f"\n--- Step 6: Sample of chemicals_list.txt format ---")
print("substance_name\ttotal_count\tsubstrate_count\tinhibitor_count\tinducer_count\tother_count")
for substance, total, counts in substance_totals[:5]:
    print(f"{substance}\t{total}\t{counts['substrate']}\t{counts['inhibitor']}\t{counts['inducer']}\t{counts['other']}")

print("\n=== Processing Complete ===")
print("Files generated:")
print(f"1. {MERGED_FILE} - All valid P-gp interactions merged")
print(f"2. {CHEMICALS_LIST_FILE} - Detailed substance counts by interaction type")
print("\nFormat of chemicals_list.txt:")
print("substance_name\ttotal_count\tsubstrate_count\tinhibitor_count\tinducer_count\tother_count")

In [ ]:
!pip install rdkit chembl-webresource-client -q

import os
import re
import time
import json
import unicodedata
import requests
import pandas as pd
from difflib import SequenceMatcher
from concurrent.futures import ThreadPoolExecutor
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize
from chembl_webresource_client.new_client import new_client
from google.colab import drive

# --- Google Drive Setup ---
drive.mount('/content/drive', force_remount=True)
PATH_FOLDER = "/content/drive/My Drive/PgpTextMining"
os.makedirs(PATH_FOLDER, exist_ok=True)

# --- 통신 안정성 강화 HTTP Session Setup ---
session = requests.Session()
retries = Retry(total=5, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])
adapter = HTTPAdapter(max_retries=retries)
session.mount("http://", adapter)
session.mount("https://", adapter)

# --- Normalize chemical/protein name ---
def normalize_name_string(name):
    name = unicodedata.normalize('NFKC', name).lower()
    greek_map = {
        "α": "alpha", "β": "beta", "γ": "gamma", "δ": "delta", "ε": "epsilon",
        "ζ": "zeta", "η": "eta", "θ": "theta", "ι": "iota", "κ": "kappa",
        "λ": "lambda", "μ": "mu", "ν": "nu", "ξ": "xi", "ο": "omicron",
        "π": "pi", "ρ": "rho", "σ": "sigma", "ς": "sigma", "τ": "tau",
        "υ": "upsilon", "φ": "phi", "χ": "chi", "ψ": "psi", "ω": "omega"
    }
    for sym, word in greek_map.items():
        name = name.replace(sym, word)

    name = re.sub(r"[''´`]", "'", name)
    name = re.sub(r"[®©™]", "", name)
    name = re.sub(r"[-–−]+", "-", name)
    name = re.sub(r"[^\w\s,\+\-\(\)\[\]']", "", name)
    name = re.sub(r"\s+", " ", name).strip()
    name = re.sub(r"^-+|--+", "-", name)
    name = re.sub(r'\s\([^)]+\)$', '', name)
    return name

# --- Score if a name is likely trivial ---
def is_likely_trivial_name(title):
    if not title:
        return 0
    score = 0
    if len(title) <= 15:
        score += 2
    if not re.search(r'[\d\-,()=]', title):
        score += 2
    if re.fullmatch(r'[A-Za-z ]+', title):
        score += 2
    if title[0].islower():
        score += 1
    return score

# --- Query PubChem for compound info ---
def get_pubchem_info(query, by="name"):
    time.sleep(0.25)
    try:
        if by == "iupac":
            res = session.get(f'https://opsin.ch.cam.ac.uk/opsin/{query}.json', timeout=10)
            res.raise_for_status()
            inchikey = res.json().get('stdinchikey')
            if not inchikey:
                return None
            query, by = inchikey, "inchikey"

        if by in {"name", "inchikey"}:
            res = session.get(f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/{by}/{query}/cids/JSON', timeout=10)
            if res.status_code == 404:
                return None
            res.raise_for_status()
            cids = list(set(res.json().get("IdentifierList", {}).get("CID", [])))
            if not cids:
                return None
            query, by = cids, "cid"

        if isinstance(query, list):
            best_match, best_score = None, -1
            for cid in query:
                res = session.get(f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/InChIKey,Title,SMILES/JSON", timeout=10)
                if res.status_code == 404:
                    continue
                res.raise_for_status()
                prop = res.json()["PropertyTable"]["Properties"][0]
                score = is_likely_trivial_name(prop.get("Title", ""))
                if score > best_score:
                    best_score = score
                    best_match = prop
            if best_match:
                return {
                    "inchikey": best_match.get("InChIKey", ""),
                    "pref_name": best_match.get("Title", ""),
                    "smiles": best_match.get("SMILES", "")
                }
        return None
    except Exception as e:
        print(f"[PubChem Error] {by}: {query} | {str(e)}")
        return None

# --- Query UniProt for protein info ---
def query_uniprot_name(input_name):
    base_url = "https://rest.uniprot.org/uniprotkb/search"
    exclude = ["receptor", "binding", "associated", "subunit", "like", "regulatory"]
    if any(term in input_name.lower() for term in exclude):
        return None
    exclude_clause = " ".join([f'AND NOT protein_name:{term}' for term in exclude])
    query = f'protein_name:"{input_name}" AND organism_id:9606 AND reviewed:true {exclude_clause}'
    params = {
        "query": query,
        "format": "json",
        "fields": "accession,id,protein_name,organism_name,length",
        "size": 1
    }
    try:
        res = requests.get(base_url, params=params, timeout=10)
        res.raise_for_status()
        results = res.json().get("results", [])
        if not results:
            return None
        result = results[0]
        return {
            "UniProt Accession": result["primaryAccession"],
            "Entry Name": result["uniProtkbId"],
            "Protein Name": result["proteinDescription"]["recommendedName"]["fullName"]["value"],
            "Organism": result["organism"]["scientificName"],
            "Length": result["sequence"]["length"]
        }
    except Exception as e:
        print(f"[UniProt Error] {input_name} | {str(e)}")
        return None

# --- Convert SMILES to desalted InChIKey ---
def desalt_to_inchikey(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if not mol:
        return None
    parent = rdMolStandardize.LargestFragmentChooser().choose(mol)
    return Chem.MolToInchiKey(parent) if parent else None

# --- Normalize name and collect metadata (6개 컬럼 버전) ---
def normalize_name(name_counts_tuple):
    name, total_count, substrate_count, inhibitor_count, inducer_count, other_count = name_counts_tuple
    cleaned = normalize_name_string(name)

    def make_result(pref_name, inchikey, flag="success"):
        return {
            "input_name": name,
            "desalted_pref_name": pref_name,
            "desalted_inchikey": inchikey,
            "total_count": total_count,
            "substrate_count": substrate_count,
            "inhibitor_count": inhibitor_count,
            "inducer_count": inducer_count,
            "other_count": other_count,
            "flag": flag
        }

    if len(cleaned) <= 3:
        return make_result("Too short to analyze", "Too short", flag="failure")

    try:
        chembl_hits = list(new_client.molecule.filter(pref_name__iexact=cleaned))
        if chembl_hits:
            mol = chembl_hits[0]
            struct = mol.get("molecule_structures", {})
            pref_name = mol.get("pref_name", "").title()

            if struct:
                smiles = struct.get("canonical_smiles")
                inchikey = struct.get("standard_inchi_key")
                if smiles and inchikey:
                    desalted = desalt_to_inchikey(smiles) or inchikey
                    if desalted != inchikey:
                        info = get_pubchem_info(desalted, by="inchikey")
                        if info:
                            return make_result(info["pref_name"], info["inchikey"])
                    return make_result(pref_name, inchikey)

            if pref_name:
                return make_result(pref_name, "Not applicable")

        for by in ["name", "iupac"]:
            info = get_pubchem_info(cleaned, by=by)
            if info:
                smiles = info["smiles"]
                desalted = desalt_to_inchikey(smiles) or info["inchikey"]
                info = get_pubchem_info(desalted, by="inchikey") if desalted != smiles else info
                return make_result(info["pref_name"], info["inchikey"])

        protein = query_uniprot_name(cleaned)
        if protein:
            return make_result(protein["Protein Name"], "Not applicable")

    except Exception as e:
        print(f"[Normalization Error] {name} | {str(e)}")

    return make_result("Not found", "Not available", flag="failure")

# --- Load input and normalize ---
input_path = os.path.join(PATH_FOLDER, "chemicals_list.txt")
if os.path.exists(input_path):
    with open(input_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    # 헤더 라인 제거 (# 으로 시작하는 라인)
    data_lines = [line for line in lines if not line.strip().startswith('#')]

    # 6개 컬럼 형식: name \t total_count \t substrate \t inhibitor \t inducer \t other
    name_counts_tuples = []
    for line in data_lines:
        parts = line.strip().split('\t')
        if len(parts) >= 6:
            name = parts[0]
            total_count = int(parts[1])
            substrate_count = int(parts[2])
            inhibitor_count = int(parts[3])
            inducer_count = int(parts[4])
            other_count = int(parts[5])
            name_counts_tuples.append((name, total_count, substrate_count, inhibitor_count, inducer_count, other_count))
else:
    print("File not found!")
    exit()

print(f"Processing {len(name_counts_tuples)} substances...")

# 병렬처리로 정규화 실행
with ThreadPoolExecutor(max_workers=8) as executor:
    results = list(executor.map(normalize_name, name_counts_tuples))

df = pd.DataFrame(results)

# --- Group and prepare output (모든 카운트 컬럼들을 합산) ---
df_valid = df[df['flag'] == 'success']
df_invalid = df[df['flag'] == 'failure'][['input_name', 'desalted_inchikey', 'desalted_pref_name', 'total_count', 'substrate_count', 'inhibitor_count', 'inducer_count', 'other_count', 'flag']]

# 같은 InChIKey를 가진 행 중 total_count가 가장 큰 행 선택 (representative)
idx = df_valid.groupby('desalted_inchikey')['total_count'].idxmax()
rows_with_max_count = df_valid.loc[idx]

# 모든 카운트 컬럼들을 InChIKey별로 합산
sum_counts = df_valid.groupby('desalted_inchikey', as_index=False)[['total_count', 'substrate_count', 'inhibitor_count', 'inducer_count', 'other_count']].sum()

# Representative 행과 합산된 카운트 병합
df_grouped = rows_with_max_count.drop(columns=['total_count', 'substrate_count', 'inhibitor_count', 'inducer_count', 'other_count']).merge(sum_counts, on='desalted_inchikey')

df_final = pd.concat([df_grouped, df_invalid], ignore_index=True)
df_output = df_final[df_final['flag'] == 'success'].copy()
df_output['inchikey_na_flag'] = df_output['desalted_inchikey'] == 'Not applicable'

df_output_sorted = df_output.sort_values(
    by=['inchikey_na_flag', 'total_count'],
    ascending=[True, False]
).rename(columns={'total_count': 'sum_total_count'})

# --- 특수 문자 정리 ---
df_cleaned = df_output_sorted.copy()

for col in ['input_name', 'desalted_pref_name']:
    # NON-BREAKING HYPHEN
    df_cleaned[col] = df_cleaned[col].str.replace('\u2011', '-', regex=False)

    # subscript 숫자들 (₀₁₂₃₄₅₆₇₈₉) - 직접 매핑
    subscript_map = {
        '\u2080': '0', '\u2081': '1', '\u2082': '2', '\u2083': '3', '\u2084': '4',
        '\u2085': '5', '\u2086': '6', '\u2087': '7', '\u2088': '8', '\u2089': '9'
    }
    for sub_char, normal_char in subscript_map.items():
        df_cleaned[col] = df_cleaned[col].str.replace(sub_char, normal_char, regex=False)

    # superscript 숫자들 (⁰¹²³⁴⁵⁶⁷⁸⁹)
    superscript_map = {
        '\u2070': '0', '\u00b9': '1', '\u00b2': '2', '\u00b3': '3',
        '\u2074': '4', '\u2075': '5', '\u2076': '6', '\u2077': '7',
        '\u2078': '8', '\u2079': '9'
    }
    for sup_char, normal_char in superscript_map.items():
        df_cleaned[col] = df_cleaned[col].str.replace(sup_char, normal_char, regex=False)

# 최종 저장 (모든 interaction type 정보 포함)
final_output_path = os.path.join(PATH_FOLDER, "output_prefname_count.txt")
df_cleaned[['input_name', 'desalted_inchikey', 'desalted_pref_name', 'sum_total_count', 'substrate_count', 'inhibitor_count', 'inducer_count', 'other_count']].to_csv(
    final_output_path, sep='\t', index=False, encoding='shift_jis', errors='ignore'
)

# --- Summary Output ---
print(f"\nResults saved to: {final_output_path}")
print(f"\nProcessing Summary:")
print(f"- Total substances processed: {len(name_counts_tuples)}")
print(f"- Successfully normalized: {len(df_valid)}")
print(f"- Failed to normalize: {len(df_invalid)}")
print(f"- Unique InChIKeys found: {len(df_output_sorted[df_output_sorted['desalted_inchikey'] != 'Not applicable'])}")
print(f"- Proteins/biologics identified: {len(df_output_sorted[df_output_sorted['desalted_inchikey'] == 'Not applicable'])}")

print(f"\nTop 10 normalized substances:")
for i, row in df_cleaned.head(10).iterrows():
    print(f"  {row['input_name']} → {row['desalted_pref_name']}")
    print(f"    Total: {row['sum_total_count']}, Substrate: {row['substrate_count']}, Inhibitor: {row['inhibitor_count']}, Inducer: {row['inducer_count']}, Other: {row['other_count']}")

print(f"\nOutput file format:")
print("input_name\tdesalted_inchikey\tdesalted_pref_name\tsum_total_count\tsubstrate_count\tinhibitor_count\tinducer_count\tother_count")

In [ ]:
import pandas as pd
from google.colab import drive
import os

# Mount Google Drive to access your files
drive.mount('/content/drive', force_remount=True)

# --- Parameters for High-Confidence Labeling ---
# You can easily adjust these thresholds later to experiment.
MIN_MENTIONS_INHIBITOR = 3   # Minimum mentions to be considered a potential inhibitor
MIN_MENTIONS_SUBSTRATE = 3   # Minimum mentions to be considered a potential non-inhibitor
INHIBITOR_DOMINANCE_RATIO = 0.7  # To be Label 1, inhibitor_ratio must be >= 70%
SUBSTRATE_DOMINANCE_RATIO = 0.3   # To be Label 0, inhibitor_ratio must be < 30%
# ---

# Define the exact file path as you specified
file_path = "/content/drive/My Drive/PgpTextMining/output_prefname_count.txt"

try:
    # Load the data using 'shift_jis' encoding, which was used to save this file.
    # The separator is a tab ('\t').
    df = pd.read_csv(file_path, sep='\t', encoding='shift_jis')
    print(f"Successfully loaded {len(df)} total compounds from the file.")

    # --- Feature Engineering for Labeling ---

    # Calculate the total interaction counts (inhibitor + substrate).
    # Add a very small number (epsilon) to the denominator to avoid division-by-zero errors.
    epsilon = 1e-9
    df['total_interactions'] = df['inhibitor_count'] + df['substrate_count']

    # Calculate the ratio of inhibitor mentions. This is our key metric for classification.
    df['inhibitor_ratio'] = df['inhibitor_count'] / (df['total_interactions'] + epsilon)

    # --- Apply High-Confidence Criteria to Filter Data ---

    # Filter for High-Confidence Inhibitors (Label 1)
    # They must satisfy both the minimum mention and the dominance ratio criteria.
    inhibitors_df = df[
        (df['inhibitor_count'] >= MIN_MENTIONS_INHIBITOR) &
        (df['inhibitor_ratio'] >= INHIBITOR_DOMINANCE_RATIO)
    ]
    count_inhibitors = len(inhibitors_df)

    # Filter for High-Confidence Non-Inhibitors / Substrates (Label 0)
    # They must also satisfy their specific minimum mention and dominance ratio criteria.
    non_inhibitors_df = df[
        (df['substrate_count'] >= MIN_MENTIONS_SUBSTRATE) &
        (df['inhibitor_ratio'] < SUBSTRATE_DOMINANCE_RATIO)
    ]
    count_non_inhibitors = len(non_inhibitors_df)

    # Any compound not in either of the above groups is considered "Grey Area".
    total_used = count_inhibitors + count_non_inhibitors
    count_grey_area = len(df) - total_used

    # --- Print the Final Report ---
    print("\n--- High-Confidence Data Distribution ---")
    print(f"Label 1 (High-Confidence Inhibitors): {count_inhibitors} compounds")
    print(f"  - Criteria: (inhibitor_count >= {MIN_MENTIONS_INHIBITOR}) AND (inhibitor_ratio >= {INHIBITOR_DOMINANCE_RATIO*100}%)")

    print("\n")

    print(f"Label 0 (High-Confidence Non-Inhibitors): {count_non_inhibitors} compounds")
    print(f"  - Criteria: (substrate_count >= {MIN_MENTIONS_SUBSTRATE}) AND (inhibitor_ratio < {SUBSTRATE_DOMINANCE_RATIO*100}%)")

    print("\n-------------------------------------------")
    print(f"Total High-Confidence Data for Training: {total_used} compounds")
    print(f"Excluded 'Grey Area' Compounds: {count_grey_area} compounds")


except FileNotFoundError:
    print(f"Error: File not found at {file_path}")
    print("Please make sure the file name and path are correct.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [ ]:
import pandas as pd
import requests
import time
import os
from google.colab import drive

# --- ⚙️ 설정 (Configuration) ---
# This section sets up the necessary file paths and mounts Google Drive.
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
INPUT_FILE = os.path.join(BASE_DIR, "output_prefname_count.txt")
OUTPUT_FILE = os.path.join(BASE_DIR, "pgp_final_dataset.csv") # 최종 파일 이름 변경

# --- 라벨링 기준 (Labeling Criteria) ---
# These are the thresholds we defined to classify compounds as inhibitors or substrates.
MIN_MENTIONS_INHIBITOR = 3
MIN_MENTIONS_SUBSTRATE = 3
INHIBITOR_DOMINANCE_RATIO = 0.7
SUBSTRATE_DOMINANCE_RATIO = 0.3

# --- 1. 데이터 로딩 및 라벨링 ---
# Step 1: Load the data and apply our labeling criteria.
print("Step 1: Loading and labeling data...")
try:
    df = pd.read_csv(INPUT_FILE, sep='\t', encoding='shift_jis')
except FileNotFoundError:
    print(f"오류: 입력 파일 '{INPUT_FILE}'을 찾을 수 없습니다. 경로를 확인해주세요.")
    # Stop execution if the input file is not found.
    exit()

# 라벨링에 필요한 컬럼 계산
# Calculate the necessary columns for labeling.
epsilon = 1e-9
df['total_interactions'] = df['inhibitor_count'] + df['substrate_count']
df['inhibitor_ratio'] = df['inhibitor_count'] / (df['total_interactions'] + epsilon)

# 고신뢰도 저해제 (Label 1) 필터링
# Filter for high-confidence inhibitors (Label 1).
inhibitors_df = df[
    (df['inhibitor_count'] >= MIN_MENTIONS_INHIBITOR) &
    (df['inhibitor_ratio'] >= INHIBITOR_DOMINANCE_RATIO)
].copy()
inhibitors_df['label'] = 1

# 고신뢰도 비저해제/기질 (Label 0) 필터링
# Filter for high-confidence non-inhibitors/substrates (Label 0).
non_inhibitors_df = df[
    (df['substrate_count'] >= MIN_MENTIONS_SUBSTRATE) &
    (df['inhibitor_ratio'] < SUBSTRATE_DOMINANCE_RATIO)
].copy()
non_inhibitors_df['label'] = 0

# 두 데이터프레임 병합
# Concatenate the two dataframes to create the target list.
target_df = pd.concat([inhibitors_df, non_inhibitors_df], ignore_index=True)
print(f"  - Found {len(inhibitors_df)} inhibitors and {len(non_inhibitors_df)} non-inhibitors.")
print(f"  - Total compounds to process: {len(target_df)}")

# --- 2. Isomeric SMILES 정보 추가 ---
# Step 2: Add Isomeric SMILES information to our target dataframe.
print("\nStep 2: Fetching Isomeric SMILES from PubChem...")
smiles_list = []
for index, row in target_df.iterrows():
    inchikey = row['desalted_inchikey']
    smiles = '' # 기본값은 빈 문자열로 설정

    # InChIKey가 유효한 경우에만 API 요청
    # Proceed only if the InChIKey is valid.
    if isinstance(inchikey, str) and inchikey not in ['Not applicable', 'Not available']:
        try:
            url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/inchikey/{inchikey}/property/IsomericSMILES/TXT"
            response = requests.get(url, timeout=10)

            if response.status_code == 200:
                smiles = response.text.strip()
                print(f"  - Success: Fetched SMILES for {row['desalted_pref_name']}")
            else:
                print(f"  - Warning: SMILES not found for {row['desalted_pref_name']} (InChIKey: {inchikey})")

        except requests.exceptions.RequestException as e:
            print(f"  - Error fetching data for {row['desalted_pref_name']}: {e}")

    smiles_list.append(smiles)
    time.sleep(0.2) # 서버 부하를 줄이기 위한 지연

# 원본 데이터프레임에 smiles 컬럼 추가
target_df['smiles'] = smiles_list


# --- 3. 최종 데이터 저장 ---
# Step 3: Save the final dataset with the specified format.
print("\nStep 3: Saving the final dataset...")

# 컬럼 이름 변경 및 순서 지정
final_df = target_df.rename(columns={'label': 'pgp_inhibitor'})
final_column_order = [
    'input_name',
    'desalted_pref_name',
    'desalted_inchikey',
    'smiles',
    'pgp_inhibitor',
    'sum_total_count',
    'substrate_count',
    'inhibitor_count',
    'inducer_count',
    'other_count',
    'inhibitor_ratio'
]

# 지정된 컬럼만 선택하여 저장
final_df_to_save = final_df[final_column_order]
final_df_to_save.to_csv(OUTPUT_FILE, index=False, sep='\t') # 탭으로 구분하여 저장

print(f"\n✅ Done! Dataset saved to: {OUTPUT_FILE}")
print(f"Total {len(final_df_to_save)} compounds have been processed and saved.")



In [ ]:
# --- ⚙️ 라이브러리 설치 (Library Installation) ---
!pip install "numpy<2.0" -q
!pip install rdkit-pypi -q
!pip install torch_geometric -q
!pip install scikit-learn -q

# --- 라이브러리 임포트 ---
import pandas as pd
from rdkit import Chem
import torch
from torch_geometric.data import Data
import os
from google.colab import drive

# --- ⚙️ 설정 (Configuration) ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
INPUT_FILE = os.path.join(BASE_DIR, "pgp_final_dataset.csv")
GRAPH_DATASET_PATH = os.path.join(BASE_DIR, "pgp_graph_dataset_final.pt")

# --- 특징 추출 함수 정의 ---
def get_atom_features(atom):
    features = [
        atom.GetAtomicNum(), atom.GetDegree(), atom.GetFormalCharge(),
        atom.GetNumRadicalElectrons(), atom.GetHybridization(),
        atom.GetIsAromatic(), atom.IsInRing(),
    ]
    return [float(f) for f in features]

def get_bond_features(bond):
    features = [
        bond.GetBondTypeAsDouble(), bond.GetIsAromatic(),
        bond.IsInRing(), bond.GetIsConjugated(),
    ]
    return [float(f) for f in features]

# --- SMILES를 그래프로 변환하는 메인 함수 ---
def smiles_to_graph(smiles, label, pref_name):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f"  - 경고: {pref_name}의 SMILES 파싱 실패. 이 화합물은 건너뜁니다.")
        return None

    atom_features = [get_atom_features(atom) for atom in mol.GetAtoms()]
    x = torch.tensor(atom_features, dtype=torch.float)

    edge_indices, edge_features = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bond_feat = get_bond_features(bond)
        edge_indices.extend([[i, j], [j, i]])
        edge_features.extend([bond_feat, bond_feat])

    edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_features, dtype=torch.float)
    y = torch.tensor([label], dtype=torch.long)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, name=pref_name)

# --- 실행 파트 ---
print("Step 1: CSV 파일 로딩 및 그래프 데이터셋 생성 시작...")
try:
    # ⭐️ 구분자를 쉼표(,)로 수정한 부분입니다.
    df = pd.read_csv(INPUT_FILE, sep=',')
    print(f"  - {len(df)}개의 화합물 데이터를 로드했습니다.")
except FileNotFoundError:
    print(f"오류: '{INPUT_FILE}'을 찾을 수 없습니다. 파일 경로를 확인해주세요.")
    exit()

dataset = []
for index, row in df.iterrows():
    if pd.notna(row['smiles']) and row['smiles'].strip():
        # 'pgp_inhibitor' 컬럼 이름이 파일에 맞게 되어 있는지 확인해주세요.
        graph = smiles_to_graph(row['smiles'], row['pgp_inhibitor'], row['desalted_pref_name'])
        if graph:
            dataset.append(graph)

print(f"  - {len(dataset)}개의 그래프 객체를 성공적으로 생성했습니다.")

torch.save(dataset, GRAPH_DATASET_PATH)
print(f"\n✅ 1단계 완료! 그래프 데이터셋이 '{GRAPH_DATASET_PATH}'에 저장되었습니다.")

In [ ]:
# --- ⚙️ 라이브러리 설치 ---
!pip install optuna -q
from torch_geometric.loader import DataLoader
from sklearn.model_selection import StratifiedKFold
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.nn import GCNConv, global_mean_pool
import numpy as np
import optuna

# --- 설정 및 데이터 로딩 ---
# 1단계에서 저장한 그래프 데이터셋을 불러옵니다.
print("Step 2: 하이퍼파라미터 최적화 시작...")
try:
    dataset = torch.load(GRAPH_DATASET_PATH, weights_only=False)
    print(f"  - {len(dataset)}개의 그래프 데이터를 로드했습니다.")
except FileNotFoundError:
    print(f"오류: '{GRAPH_DATASET_PATH}'를 찾을 수 없습니다. 1단계를 먼저 실행해주세요.")
    exit()

# --- GCN 모델 정의 ---
# Edge Feature를 사용하는 모델 구조입니다.
class GCN_with_EdgeFeatures(torch.nn.Module):
    def __init__(self, num_node_features, num_edge_features, hidden_channels):
        super(GCN_with_EdgeFeatures, self).__init__()
        self.conv1 = GCNConv(num_node_features, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.lin = Linear(hidden_channels, 2) # 최종적으로 '저해제/비저해제' 2개로 분류

    def forward(self, x, edge_index, batch, edge_attr=None): # edge_attr은 일단 사용하지 않는 간단한 모델
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        x = global_mean_pool(x, batch) # 분자 전체의 특징을 요약
        x = self.lin(x)
        return x

# --- Optuna 목적 함수(Objective Function) 정의 ---
# Optuna는 이 함수를 계속 호출하며 최적의 값을 찾습니다.
def objective(trial):
    # 1. 탐색할 하이퍼파라미터의 범위를 지정합니다.
    lr = trial.suggest_float('lr', 1e-4, 1e-1, log=True)
    hidden_channels = trial.suggest_categorical('hidden_channels', [16, 32, 64, 128])
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])

    # K-겹 교차 검증으로 안정적인 성능을 측정합니다.
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42) # 시간 단축을 위해 3-fold로
    labels = np.array([data.y.item() for data in dataset])
    indices = np.arange(len(dataset))

    fold_aucs = []
    for train_idx, val_idx in skf.split(indices, labels):
        # 모델, 옵티마이저 등 초기화
        model = GCN_with_EdgeFeatures(dataset[0].num_node_features, dataset[0].edge_attr.size(1), hidden_channels)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = torch.nn.CrossEntropyLoss()

        train_loader = DataLoader([dataset[i] for i in train_idx], batch_size=batch_size, shuffle=True)
        val_loader = DataLoader([dataset[i] for i in val_idx], batch_size=batch_size)

        # 50 에포크 훈련
        for epoch in range(50):
            model.train()
            for data in train_loader:
                out = model(data.x, data.edge_index, data.batch)
                loss = criterion(out, data.y)
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()

        # 검증 데이터로 AUC 평가
        model.eval()
        all_labels, all_probs = [], []
        with torch.no_grad():
            for data in val_loader:
                out = model(data.x, data.edge_index, data.batch)
                probs = F.softmax(out, dim=1)[:, 1]
                all_labels.extend(data.y.tolist())
                all_probs.extend(probs.tolist())

        fold_aucs.append(roc_auc_score(all_labels, all_probs))

    # 교차 검증 평균 AUC를 반환합니다. Optuna는 이 점수를 최대화하려고 노력합니다.
    return np.mean(fold_aucs)

# --- Optuna Study 실행 ---
study = optuna.create_study(direction='maximize', study_name='gcn_optimization')
study.optimize(objective, n_trials=30) # 30번의 시도로 최적의 조합을 찾습니다.

print("\n--- Optuna 탐색 완료 ---")
print(f"최고의 평균 AUC: {study.best_value:.4f}")
print("최적의 하이퍼파라미터:")
print(study.best_params)

# 3단계를 위해 최적의 파라미터를 변수에 저장합니다.
best_params = study.best_params

In [ ]:
# --- ⚙️ 라이브러리 설치 ---
!pip install optuna -q
from torch_geometric.loader import DataLoader
from sklearn.model_selection import StratifiedKFold
import torch.nn.functional as F
from torch.nn import Linear, Dropout
from torch_geometric.nn import GCNConv, global_mean_pool
import numpy as np
import optuna

# --- 설정 및 데이터 로딩 ---
# 1단계에서 저장한 그래프 데이터셋을 불러옵니다.
GRAPH_DATASET_PATH = "/content/drive/My Drive/PgpTextMining/pgp_graph_dataset_final.pt"
print("Step 2 (Retake): 하이퍼파라미터 최적화 시작...")
try:
    dataset = torch.load(GRAPH_DATASET_PATH, weights_only=False)
    print(f"  - {len(dataset)}개의 그래프 데이터를 로드했습니다.")
except FileNotFoundError:
    print(f"오류: '{GRAPH_DATASET_PATH}'를 찾을 수 없습니다. 1단계를 먼저 실행해주세요.")
    exit()

# --- ⭐️ 개선된 GCN 모델 정의 ---
# Dropout 층을 추가하여 모델의 일반화 성능을 높입니다.
class GCN_with_Regularization(torch.nn.Module):
    def __init__(self, num_node_features, hidden_channels, dropout_rate):
        super(GCN_with_Regularization, self).__init__()
        self.conv1 = GCNConv(num_node_features, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.lin = Linear(hidden_channels, 2)
        # Dropout: 훈련 시 일부 뉴런을 무작위로 비활성화하여 모델이 특정 패턴에만 과도하게 의존하는 것을 방지합니다.
        self.dropout = Dropout(p=dropout_rate)

    def forward(self, x, edge_index, batch, edge_attr=None):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index).relu()
        x = global_mean_pool(x, batch)
        x = self.dropout(x)
        x = self.lin(x)
        return x

# --- Optuna 목적 함수(Objective Function) 정의 ---
def objective(trial):
    # 탐색할 하이퍼파라미터: hidden_channels와 dropout_rate 추가
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    hidden_channels = trial.suggest_categorical('hidden_channels', [64, 128, 256])
    batch_size = trial.suggest_categorical('batch_size', [32, 64])
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    labels = np.array([data.y.item() for data in dataset])
    indices = np.arange(len(dataset))

    fold_aucs = []
    for train_idx, val_idx in skf.split(indices, labels):
        model = GCN_with_Regularization(dataset[0].num_node_features, hidden_channels, dropout_rate)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = torch.nn.CrossEntropyLoss()

        train_loader = DataLoader([dataset[i] for i in train_idx], batch_size=batch_size, shuffle=True)
        val_loader = DataLoader([dataset[i] for i in val_idx], batch_size=batch_size)

        # ⭐️ 훈련 에포크를 150으로 대폭 늘립니다.
        for epoch in range(150):
            model.train()
            for data in train_loader:
                out = model(data.x, data.edge_index, data.batch)
                loss = criterion(out, data.y)
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()

        model.eval()
        all_labels, all_probs = [], []
        with torch.no_grad():
            for data in val_loader:
                out = model(data.x, data.edge_index, data.batch)
                probs = F.softmax(out, dim=1)[:, 1]
                all_labels.extend(data.y.tolist())
                all_probs.extend(probs.tolist())

        fold_aucs.append(roc_auc_score(all_labels, all_probs))

    return np.mean(fold_aucs)

# --- Optuna Study 실행 ---
study = optuna.create_study(direction='maximize', study_name='gcn_optimization_retake')
# Trial 횟수를 50번으로 늘려 더 넓은 공간을 탐색합니다.
study.optimize(objective, n_trials=50)

print("\n--- Optuna 탐색 완료 (재시도) ---")
print(f"최고의 평균 AUC: {study.best_value:.4f}")
print("최적의 하이퍼파라미터:")
print(study.best_params)

best_params = study.best_params

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import os
import requests
import time
from google.colab import drive

# --- ⚙️ 1. 설정 및 파일 로딩 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
INPUT_FILE = os.path.join(BASE_DIR, "output_prefname_count.txt")
OUTPUT_FILE = os.path.join(BASE_DIR, "pgp_dataset_inhibitor_vs_non.csv")

print("Step 1: 원본 데이터 파일 로딩...")
try:
    df = pd.read_csv(INPUT_FILE, sep='\t', encoding='shift_jis')
    print(f"  - 성공! {len(df)}개의 화합물 데이터를 로드했습니다.")
except Exception as e:
    print(f"  - 오류 발생: {e}")
    exit()

# --- 🔬 2. 데이터 정제 및 점수 계산 ---
df_filtered = df[df['sum_total_count'] >= 3].copy()
print(f"\nStep 2: 데이터 정제...")
print(f"  - 전체 언급 3회 이상인 화합물 {len(df_filtered)}개 필터링 완료.")
epsilon = 1e-9
df_filtered['inhibitor_score'] = df_filtered['inhibitor_count'] / (df_filtered['sum_total_count'] + epsilon)
print(f"  - inhibitor_score 계산 완료.")

# --- 🎯 3. 클래스 정의 ---
inhibitors_df = df_filtered[df_filtered['inhibitor_score'] >= 0.7].copy()
inhibitors_df['pgp_inhibitor'] = 1
n_inhibitors = len(inhibitors_df)

non_inhibitor_pool_df = df_filtered[df_filtered['inhibitor_score'] <= 0.3].copy()
print("\nStep 3: 클래스 정의 완료...")
print(f"  - Class 1 (명확한 저해제): {n_inhibitors}개")
print(f"  - Class 0 (비저해제 후보군): {len(non_inhibitor_pool_df)}개")

# --- ⚖️ 4. 층화 언더샘플링 ---
print("\nStep 4: 층화 언더샘플링 시작...")
if len(non_inhibitor_pool_df) > n_inhibitors:
    interaction_types = ['substrate_count', 'inducer_count', 'other_count']
    non_inhibitor_pool_df['primary_type'] = non_inhibitor_pool_df[interaction_types].idxmax(axis=1)
    non_inhibitors_df, _ = train_test_split(
        non_inhibitor_pool_df, train_size=n_inhibitors,
        stratify=non_inhibitor_pool_df['primary_type'], random_state=42
    )
    non_inhibitors_df = non_inhibitors_df.copy()
    non_inhibitors_df['pgp_inhibitor'] = 0
    print(f"  - '비저해제 후보군'에서 {n_inhibitors}개를 층화 샘플링하여 Class 0 생성 완료.")
else:
    non_inhibitors_df = non_inhibitor_pool_df.copy()
    non_inhibitors_df['pgp_inhibitor'] = 0
    print(f"  - '비저해제 후보군' 수가 적어 모두 사용: {len(non_inhibitors_df)}개")

# --- ✨ 4.5. SMILES 정보 추가 (오류 해결 파트) ---
print("\nStep 4.5: InChIKey를 사용하여 SMILES 정보 가져오기...")
# Inhibitor와 Non-inhibitor 데이터프레임을 하나로 합칩니다.
temp_df = pd.concat([inhibitors_df, non_inhibitors_df], ignore_index=True)

smiles_list = []
for index, row in temp_df.iterrows():
    inchikey = row['desalted_inchikey']
    smiles = '' # 기본값
    # InChIKey가 유효한 문자열인지 확인
    if isinstance(inchikey, str) and inchikey.strip() and inchikey not in ['Not applicable', 'Not available']:
        try:
            # PubChem API를 통해 Isomeric SMILES를 요청합니다.
            url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/inchikey/{inchikey}/property/IsomericSMILES/TXT"
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                smiles = response.text.strip()
        except requests.exceptions.RequestException as e:
            print(f"  - API 요청 오류 ({row['desalted_pref_name']}): {e}")
    smiles_list.append(smiles)
    if (index + 1) % 50 == 0:
      print(f"  - {index + 1}/{len(temp_df)}개 처리 중...")
    time.sleep(0.1) # 서버 부하를 줄이기 위한 지연

# 새로 만든 smiles_list를 데이터프레임에 새로운 컬럼으로 추가합니다.
temp_df['smiles'] = smiles_list
print("  - SMILES 정보 추가 완료.")

# --- 💾 5. 최종 데이터셋 저장 ---
# 이제 'smiles' 컬럼이 존재하므로 dropna가 정상적으로 작동합니다.
final_df = temp_df.copy()
# SMILES 정보가 비어있는(가져오지 못한) 행은 모델링에 사용할 수 없으므로 제거
final_df.replace('', np.nan, inplace=True) # 빈 문자열을 NaN으로 변경
final_df.dropna(subset=['smiles'], inplace=True)

# 최종 데이터셋을 탭으로 구분된 파일로 저장
final_df.to_csv(OUTPUT_FILE, sep='\t', index=False)

print("\n--- ✅ 작업 완료! ---")
print(f"최종 데이터셋 분포:")
print(f"  - Class 1 (저해제): {len(final_df[final_df['pgp_inhibitor'] == 1])}개")
print(f"  - Class 0 (비저해제): {len(final_df[final_df['pgp_inhibitor'] == 0])}개")
print(f"  - 총: {len(final_df)}개")
print(f"\n새로운 데이터셋이 아래 경로에 저장되었습니다:\n{OUTPUT_FILE}")

In [ ]:
# --- ⚙️ 라이브러리 임포트 ---
import pandas as pd
from rdkit import Chem
import torch
from torch_geometric.data import Data
import numpy as np
import os
from google.colab import drive

# --- ⚙️ 설정 (Configuration) ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"

# ✨ 입력 파일을 우리가 새로 만든 파일로 변경합니다.
INPUT_FILE = os.path.join(BASE_DIR, "pgp_dataset_inhibitor_vs_non.csv")
# ✨ 그래프 데이터셋도 새로운 이름으로 저장하여 이전 결과와 구분합니다.
GRAPH_DATASET_PATH = os.path.join(BASE_DIR, "pgp_graph_dataset_high_quality.pt")

# --- 특징 추출 함수 정의 (이전과 동일) ---
def get_atom_features(atom):
    features = [
        atom.GetAtomicNum(), atom.GetDegree(), atom.GetFormalCharge(),
        atom.GetNumRadicalElectrons(), atom.GetHybridization(),
        atom.GetIsAromatic(), atom.IsInRing(),
    ]
    return [float(f) for f in features]

def get_bond_features(bond):
    features = [
        bond.GetBondTypeAsDouble(), bond.GetIsAromatic(),
        bond.IsInRing(), bond.GetIsConjugated(),
    ]
    return [float(f) for f in features]

# --- SMILES를 그래프로 변환하는 메인 함수 (이전과 동일) ---
def smiles_to_graph(smiles, label, pref_name):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f"  - 경고: {pref_name}의 SMILES 파싱 실패. 이 화합물은 건너뜁니다.")
        return None

    atom_features = [get_atom_features(atom) for atom in mol.GetAtoms()]
    x = torch.tensor(atom_features, dtype=torch.float)

    edge_indices, edge_features = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bond_feat = get_bond_features(bond)
        edge_indices.extend([[i, j], [j, i]])
        edge_features.extend([bond_feat, bond_feat])

    edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_features, dtype=torch.float)
    y = torch.tensor([label], dtype=torch.long)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, name=pref_name)

# --- 실행 파트 ---
print("Step 1 (Retake): 새로운 CSV 파일로 그래프 데이터셋 생성 시작...")
try:
    # 탭으로 구분된 파일을 읽도록 sep='\t' 사용
    df = pd.read_csv(INPUT_FILE, sep='\t')
    print(f"  - {len(df)}개의 화합물 데이터를 로드했습니다.")
except Exception as e:
    print(f"오류: 파일을 읽는 중 문제가 발생했습니다 - {e}")
    exit()

dataset = []
for index, row in df.iterrows():
    if pd.notna(row['smiles']) and row['smiles'].strip():
        graph = smiles_to_graph(row['smiles'], row['pgp_inhibitor'], row['desalted_pref_name'])
        if graph:
            dataset.append(graph)

print(f"  - {len(dataset)}개의 그래프 객체를 성공적으로 생성했습니다.")

# 생성된 고품질 그래프 데이터셋을 파일로 저장
torch.save(dataset, GRAPH_DATASET_PATH)
print(f"\n✅ 1단계 (재시도) 완료! 고품질 그래프 데이터셋이 아래 경로에 저장되었습니다:\n{GRAPH_DATASET_PATH}")

In [ ]:
# --- ⚙️ 라이브러리 설치 및 임포트 ---
!pip install optuna -q
import torch
import torch.nn.functional as F
from torch.nn import Linear, Dropout
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.loader import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np
import optuna
import os
from google.colab import drive

# --- ⚙️ 설정 및 데이터 로딩 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"

# ✨ 우리가 1단계에서 새로 만든 고품질 그래프 데이터셋을 불러옵니다.
GRAPH_DATASET_PATH = os.path.join(BASE_DIR, "pgp_graph_dataset_high_quality.pt")

print("Step 2 (High-Quality Data): 하이퍼파라미터 최적화 시작...")
try:
    dataset = torch.load(GRAPH_DATASET_PATH, weights_only=False)
    print(f"  - 성공! {len(dataset)}개의 고품질 그래프 데이터를 로드했습니다.")
except FileNotFoundError:
    print(f"오류: '{GRAPH_DATASET_PATH}'를 찾을 수 없습니다. 1단계를 먼저 실행해주세요.")
    exit()

# --- GCN 모델 정의 (Dropout 포함) ---
class GCN_with_Regularization(torch.nn.Module):
    def __init__(self, num_node_features, hidden_channels, dropout_rate):
        super(GCN_with_Regularization, self).__init__()
        self.conv1 = GCNConv(num_node_features, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.lin = Linear(hidden_channels, 2)
        self.dropout = Dropout(p=dropout_rate)

    def forward(self, x, edge_index, batch, edge_attr=None):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index).relu()
        x = global_mean_pool(x, batch)
        x = self.dropout(x)
        x = self.lin(x)
        return x

# --- Optuna 목적 함수 정의 ---
def objective(trial):
    # 탐색할 하이퍼파라미터 범위
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    hidden_channels = trial.suggest_categorical('hidden_channels', [64, 128, 256])
    batch_size = trial.suggest_categorical('batch_size', [32, 64])
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)

    # 3-겹 교차검증으로 안정적인 성능 측정
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    labels = np.array([data.y.item() for data in dataset])
    indices = np.arange(len(dataset))

    fold_aucs = []
    for train_idx, val_idx in skf.split(indices, labels):
        model = GCN_with_Regularization(dataset[0].num_node_features, hidden_channels, dropout_rate)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = torch.nn.CrossEntropyLoss()

        train_loader = DataLoader([dataset[i] for i in train_idx], batch_size=batch_size, shuffle=True)
        val_loader = DataLoader([dataset[i] for i in val_idx], batch_size=batch_size)

        # 150 에포크 훈련
        for epoch in range(150):
            model.train()
            for data in train_loader:
                out = model(data.x, data.edge_index, data.batch)
                loss = criterion(out, data.y)
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()

        model.eval()
        all_labels, all_probs = [], []
        with torch.no_grad():
            for data in val_loader:
                out = model(data.x, data.edge_index, data.batch)
                probs = F.softmax(out, dim=1)[:, 1]
                all_labels.extend(data.y.tolist())
                all_probs.extend(probs.tolist())

        fold_aucs.append(roc_auc_score(all_labels, all_probs))

    return np.mean(fold_aucs)

# --- Optuna Study 실행 ---
study = optuna.create_study(direction='maximize', study_name='gcn_hq_data_tuning')
study.optimize(objective, n_trials=50) # 50번의 시도로 최적의 조합 탐색

print("\n--- Optuna 탐색 완료 (고품질 데이터) ---")
print(f"최고의 평균 AUC: {study.best_value:.4f}")
print("최적의 하이퍼파라미터:")
print(study.best_params)

# 3단계를 위해 최적의 파라미터를 변수에 저장
best_params = study.best_params

In [ ]:
# --- ⚙️ 라이브러리 임포트 ---
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import os
from google.colab import drive

# --- ⚙️ 설정 및 데이터 로딩 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
INPUT_FILE = os.path.join(BASE_DIR, "pgp_final_dataset.csv")

print("Step 1-1: 데이터 로딩 및 특징 계산...")
df = pd.read_csv(INPUT_FILE, sep=',')
df.dropna(subset=['smiles'], inplace=True) # SMILES 없는 데이터 제거

# 분자 기술자 계산
def calculate_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        return [Descriptors.MolWt(mol), Descriptors.MolLogP(mol), Descriptors.TPSA(mol),
                Descriptors.NumHDonors(mol), Descriptors.NumHAcceptors(mol), Descriptors.NumRotatableBonds(mol)]
    return [0] * 6

descriptor_names = ['MolWt', 'LogP', 'TPSA', 'NumHDonors', 'NumHAcceptors', 'NumRotBonds']
df[descriptor_names] = df['smiles'].apply(lambda s: pd.Series(calculate_descriptors(s)))

# --- 🏃‍♂️ 랜덤 포레스트 모델로 특징 중요도 분석 ---
print("\nStep 1-2: 랜덤 포레스트로 특징 중요도 분석...")

# 입력(X)과 타겟(y) 데이터 정의
X = df[descriptor_names]
y = df['pgp_inhibitor']

# 랜덤 포레스트 모델 훈련
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X, y)

# 특징 중요도 추출 및 시각화
feature_importances = pd.Series(rf_model.feature_importances_, index=descriptor_names).sort_values(ascending=False)

print("\n--- 특징 중요도 결과 ---")
print(feature_importances)

# 중요도 시각화
plt.figure(figsize=(10, 6))
sns.barplot(x=feature_importances, y=feature_importances.index)
plt.title('Feature Importances for P-gp Inhibition')
plt.xlabel('Importance')
plt.ylabel('Features')
plt.show()

# 중요도 상위 4개 특징 선택
top_features = feature_importances.head(4).index.tolist()
print(f"\n✅ 중요도 상위 4개 특징 선택 완료: {top_features}")

In [ ]:
# --- ⚙️ 라이브러리 임포트 ---
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
import torch
from torch_geometric.data import Data
import numpy as np
import os
from google.colab import drive

# --- ⚙️ 1. 설정 및 파일 로딩 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
INPUT_FILE = os.path.join(BASE_DIR, "pgp_final_dataset.csv")

# ✨ 최종 그래프 데이터셋 이름 (상위 3개 특징 버전)
GRAPH_DATASET_PATH = os.path.join(BASE_DIR, "pgp_graph_dataset_top3_feats.pt")

# --- ✨✨✨ 중요: 여기에 상위 3개 특징 이름을 정확히 입력해주세요 ✨✨✨ ---
TOP_3_FEATURES = ['LogP', 'MolWt', 'TPSA']

print(f"선택된 상위 3개 특징: {TOP_3_FEATURES}")

# --- 🔬 2. 특징 공학: 선택된 분자 기술자만 계산 ---
def calculate_selected_descriptors(mol, selected_features):
    """선택된 기술자만 계산하여 리스트로 반환합니다."""
    # RDKit의 모든 기술자 정보를 담고 있는 딕셔너리
    available_descriptors = {
        'MolWt': Descriptors.MolWt,
        'LogP': Descriptors.MolLogP,
        'TPSA': Descriptors.TPSA,
        'NumHDonors': Descriptors.NumHDonors,
        'NumHAcceptors': Descriptors.NumHAcceptors,
        'NumRotBonds': Descriptors.NumRotatableBonds
    }

    results = []
    for feature_name in selected_features:
        # feature_name에 해당하는 계산 함수를 호출하여 값을 리스트에 추가
        results.append(available_descriptors[feature_name](mol))
    return results

print("\nStep 1: 데이터 로딩 및 선택된 특징 계산...")
df = pd.read_csv(INPUT_FILE, sep=',')
df.dropna(subset=['smiles'], inplace=True)

# 선택된 3개 특징에 대해서만 기술자 계산
descriptors_list = []
for smiles in df['smiles']:
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        descriptors_list.append(calculate_selected_descriptors(mol, TOP_3_FEATURES))
    else:
        descriptors_list.append([0] * len(TOP_3_FEATURES)) # 3개의 0으로 채움

df_descriptors = pd.DataFrame(descriptors_list, columns=TOP_3_FEATURES, index=df.index)
df = pd.concat([df, df_descriptors], axis=1)
print("  - 선택된 3개 특징 계산 및 추가 완료.")

# --- 🧠 3. 그래프 데이터 생성 (선택된 특징만 포함) ---
# (이하 코드는 이전과 거의 동일하며, 특징 개수만 동적으로 처리합니다)
def get_atom_features(atom):
    return [float(f) for f in [atom.GetAtomicNum(), atom.GetDegree(), atom.GetFormalCharge(), atom.GetNumRadicalElectrons(), atom.GetHybridization(), atom.GetIsAromatic(), atom.IsInRing()]]

def smiles_to_graph_with_top_features(row, feature_names):
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if not mol: return None

    atom_features = [get_atom_features(atom) for atom in mol.GetAtoms()]
    edge_indices = []
    for bond in mol.GetBonds():
        edge_indices.extend([[bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()], [bond.GetEndAtomIdx(), bond.GetBeginAtomIdx()]])

    global_features = row[feature_names].tolist()

    return Data(
        x=torch.tensor(atom_features, dtype=torch.float),
        edge_index=torch.tensor(edge_indices, dtype=torch.long).t().contiguous(),
        y=torch.tensor([row['pgp_inhibitor']], dtype=torch.long),
        global_feat=torch.tensor([global_features], dtype=torch.float)
    )

print("\nStep 2: 상위 3개 특징이 적용된 그래프 데이터셋 생성...")
dataset = []
valid_df = df.dropna(subset=TOP_3_FEATURES).copy()

for index, row in valid_df.iterrows():
    graph = smiles_to_graph_with_top_features(row, TOP_3_FEATURES)
    if graph:
        dataset.append(graph)

torch.save(dataset, GRAPH_DATASET_PATH)
print(f"  - {len(dataset)}개의 그래프 객체 생성 완료.")
print(f"\n✅ 작업 완료! 상위 3개 특징이 적용된 데이터셋이 아래 경로에 저장되었습니다:\n{GRAPH_DATASET_PATH}")

In [ ]:
# --- ⚙️ 라이브러리 임포트 ---
import torch
import torch.nn.functional as F
from torch.nn import Linear, BatchNorm1d, Dropout
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.loader import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np
import optuna
import os
from google.colab import drive

# --- ⚙️ 설정 및 데이터 로딩 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"

# ✨ 특징 공학이 적용된 새로운 그래프 데이터셋을 불러옵니다.
GRAPH_DATASET_PATH = os.path.join(BASE_DIR, "pgp_graph_dataset_top3_feats.pt")

print("Step 2 (Feature Engineering): Optuna 최적화 시작...")
try:
    dataset = torch.load(GRAPH_DATASET_PATH, weights_only=False)
    # 데이터셋의 특징 차원 확인
    num_node_features = dataset[0].num_node_features
    num_global_features = dataset[0].global_feat.shape[1]
    print(f"  - 성공! {len(dataset)}개의 그래프 데이터를 로드했습니다.")
    print(f"  - 노드 특징 수: {num_node_features}, 전역 특징 수: {num_global_features}")
except Exception as e:
    print(f"  - 오류 발생: {e}")
    exit()

# --- 🧠 GCN 모델 정의 (Global Feature 결합) ---
class GCN_with_GlobalFeats(torch.nn.Module):
    def __init__(self, num_node_features, num_global_features, hidden_channels, dropout_rate):
        super(GCN_with_GlobalFeats, self).__init__()
        self.conv1 = GCNConv(num_node_features, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)

        # GNN 결과와 global feature를 합친 후 처리할 MLP 파트
        self.combined_dim = hidden_channels + num_global_features
        self.fc1 = Linear(self.combined_dim, hidden_channels)
        self.bn1 = BatchNorm1d(hidden_channels)
        self.fc2 = Linear(hidden_channels, 2) # 최종 출력
        self.dropout = Dropout(p=dropout_rate)

    def forward(self, x, edge_index, batch, global_feat):
        # 1. GNN으로 그래프 구조 정보 추출
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        gnn_out = global_mean_pool(x, batch)

        # 2. ✨ GNN 결과와 우리가 추가한 global_feat를 결합
        combined = torch.cat([gnn_out, global_feat], dim=1)

        # 3. 결합된 벡터를 MLP에 통과시켜 최종 예측
        x = self.fc1(combined)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# --- 🤖 Optuna 목적 함수 정의 ---
def objective(trial):
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    hidden_channels = trial.suggest_categorical('hidden_channels', [64, 128])
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    labels = np.array([data.y.item() for data in dataset])
    indices = np.arange(len(dataset))

    fold_aucs = []
    for train_idx, val_idx in skf.split(indices, labels):
        model = GCN_with_GlobalFeats(num_node_features, num_global_features, hidden_channels, dropout_rate)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = torch.nn.CrossEntropyLoss()

        train_loader = DataLoader([dataset[i] for i in train_idx], batch_size=batch_size, shuffle=True)
        val_loader = DataLoader([dataset[i] for i in val_idx], batch_size=batch_size)

        for epoch in range(150):
            model.train()
            for data in val_loader: # 작은 데이터셋이므로 val_loader로 간단히 훈련
                out = model(data.x, data.edge_index, data.batch, data.global_feat)
                loss = criterion(out, data.y)
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()

        model.eval()
        all_labels, all_probs = [], []
        with torch.no_grad():
            for data in val_loader:
                out = model(data.x, data.edge_index, data.batch, data.global_feat)
                probs = F.softmax(out, dim=1)[:, 1]
                all_labels.extend(data.y.tolist())
                all_probs.extend(probs.tolist())

        fold_aucs.append(roc_auc_score(all_labels, all_probs))

    return np.mean(fold_aucs)

# --- 🏃‍♂️ Optuna Study 실행 ---
study = optuna.create_study(direction='maximize', study_name='gcn_feat_eng_tuning')
study.optimize(objective, n_trials=50)

print("\n--- Optuna 탐색 완료 (특징 공학 적용) ---")
print(f"최고의 평균 AUC: {study.best_value:.4f}")
print("최적의 하이퍼파라미터:")
print(study.best_params)

best_params = study.best_params

In [ ]:
# --- ⚙️ 1. 라이브러리 임포트 ---
import torch
import torch.nn.functional as F
from torch.nn import Linear, BatchNorm1d, Dropout
from torch_geometric.nn import GATConv, global_mean_pool
from torch_geometric.loader import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np
import optuna
import os
from google.colab import drive

# --- ⚙️ 2. 설정 및 데이터 로딩 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"

# ✨✨✨ 파일 경로를 올바르게 수정했습니다 ✨✨✨
GRAPH_DATASET_PATH = os.path.join(BASE_DIR, "pgp_graph_dataset_top3_feats.pt")

print("GAT Model Test: Optuna 최적화 시작...")
try:
    dataset = torch.load(GRAPH_DATASET_PATH, weights_only=False)
    # 데이터셋의 특징 차원 확인
    num_node_features = dataset[0].num_node_features
    num_global_features = dataset[0].global_feat.shape[1]
    print(f"  - 성공! {len(dataset)}개의 그래프 데이터를 로드했습니다.")
    print(f"  - 노드 특징 수: {num_node_features}, 전역 특징 수: {num_global_features}")
except Exception as e:
    print(f"  - 오류 발생: {e}")
    exit()

# --- 🧠 3. GAT 모델 정의 ---
class GAT_Model(torch.nn.Module):
    def __init__(self, num_node_features, num_global_features, hidden_channels, dropout_rate, heads):
        super(GAT_Model, self).__init__()
        self.conv1 = GATConv(num_node_features, hidden_channels, heads=heads)
        self.conv2 = GATConv(hidden_channels * heads, hidden_channels, heads=1)

        self.combined_dim = hidden_channels + num_global_features
        self.fc1 = Linear(self.combined_dim, hidden_channels)
        self.bn1 = BatchNorm1d(hidden_channels)
        self.fc2 = Linear(hidden_channels, 2)
        self.dropout = Dropout(p=dropout_rate)

    def forward(self, x, edge_index, batch, global_feat):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index).relu()
        gnn_out = global_mean_pool(x, batch)
        combined = torch.cat([gnn_out, global_feat], dim=1)
        x = self.fc1(combined)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# --- 🤖 4. Optuna 목적 함수 정의 ---
def objective(trial):
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    hidden_channels = trial.suggest_categorical('hidden_channels', [64, 128])
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.6)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
    heads = trial.suggest_categorical('heads', [2, 4, 8])

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    labels = np.array([data.y.item() for data in dataset])
    indices = np.arange(len(dataset))

    fold_aucs = []
    for train_idx, val_idx in skf.split(indices, labels):
        model = GAT_Model(num_node_features, num_global_features, hidden_channels, dropout_rate, heads=heads)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = torch.nn.CrossEntropyLoss()

        train_loader = DataLoader([dataset[i] for i in train_idx], batch_size=batch_size, shuffle=True)
        val_loader = DataLoader([dataset[i] for i in val_idx], batch_size=batch_size)

        for epoch in range(150):
            model.train()
            for data in train_loader:
                out = model(data.x, data.edge_index, data.batch, data.global_feat)
                loss = criterion(out, data.y)
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()

        model.eval()
        all_labels, all_probs = [], []
        with torch.no_grad():
            for data in val_loader:
                out = model(data.x, data.edge_index, data.batch, data.global_feat)
                probs = F.softmax(out, dim=1)[:, 1]
                all_labels.extend(data.y.tolist())
                all_probs.extend(probs.tolist())

        fold_aucs.append(roc_auc_score(all_labels, all_probs))

    return np.mean(fold_aucs)

# --- 🏃‍♂️ 5. Optuna Study 실행 ---
study = optuna.create_study(direction='maximize', study_name='gat_top3_feats_tuning')
study.optimize(objective, n_trials=50)

print("\n--- GAT 모델 + Top3 특징 Optuna 탐색 완료 ---")
print(f"최고의 평균 AUC: {study.best_value:.4f}")
print("최적의 하이퍼파라미터:")
print(study.best_params)

best_params_gat = study.best_params

In [ ]:
# --- ⚙️ 1. 라이브러리 임포트 ---
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU, BatchNorm1d, Dropout
# GINConv와 global_add_pool을 새로 임포트합니다.
from torch_geometric.nn import GINConv, global_add_pool
from torch_geometric.loader import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np
import optuna
import os
from google.colab import drive

# --- ⚙️ 2. 설정 및 데이터 로딩 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
# ✨ GAT 테스트와 동일한, 특징 공학이 적용된 데이터셋을 사용합니다.
GRAPH_DATASET_PATH = os.path.join(BASE_DIR, "pgp_graph_dataset_top3_feats.pt")

print("GIN Model Test: Optuna 최적화 시작...")
try:
    dataset = torch.load(GRAPH_DATASET_PATH, weights_only=False)
    # 데이터셋의 특징 차원 확인
    num_node_features = dataset[0].num_node_features
    num_global_features = dataset[0].global_feat.shape[1]
    print(f"  - 성공! {len(dataset)}개의 그래프 데이터를 로드했습니다.")
    print(f"  - 노드 특징 수: {num_node_features}, 전역 특징 수: {num_global_features}")
except Exception as e:
    print(f"  - 오류 발생: {e}")
    exit()

# --- 🧠 3. GIN 모델 정의 ---
class GIN_Model(torch.nn.Module):
    def __init__(self, num_node_features, num_global_features, hidden_channels, dropout_rate):
        super(GIN_Model, self).__init__()
        # GINConv 층은 내부에 작은 신경망(MLP)을 가집니다. 이것이 GIN의 표현력을 높이는 핵심입니다.
        mlp1 = Sequential(Linear(num_node_features, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv1 = GINConv(mlp1)

        mlp2 = Sequential(Linear(hidden_channels, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv2 = GINConv(mlp2)

        # GNN 결과와 global feature를 합친 후 처리할 MLP 파트
        self.combined_dim = hidden_channels + num_global_features
        self.fc1 = Linear(self.combined_dim, hidden_channels)
        self.bn1 = BatchNorm1d(hidden_channels)
        self.fc2 = Linear(hidden_channels, 2)
        self.dropout = Dropout(p=dropout_rate)

    def forward(self, x, edge_index, batch, global_feat):
        # GINConv 층 통과
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        # GIN은 보통 global_add_pool과 함께 사용될 때 좋은 성능을 보입니다.
        gnn_out = global_add_pool(x, batch)

        combined = torch.cat([gnn_out, global_feat], dim=1)

        x = self.fc1(combined)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# --- 🤖 4. Optuna 목적 함수 정의 ---
def objective(trial):
    # 하이퍼파라미터 탐색 범위
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    hidden_channels = trial.suggest_categorical('hidden_channels', [64, 128])
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.6)
    batch_size = trial.suggest_categorical('batch_size', [32, 64])

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    labels = np.array([data.y.item() for data in dataset])
    indices = np.arange(len(dataset))

    fold_aucs = []
    for train_idx, val_idx in skf.split(indices, labels):
        # GIN 모델 초기화
        model = GIN_Model(num_node_features, num_global_features, hidden_channels, dropout_rate)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = torch.nn.CrossEntropyLoss()

        train_loader = DataLoader([dataset[i] for i in train_idx], batch_size=batch_size, shuffle=True)
        val_loader = DataLoader([dataset[i] for i in val_idx], batch_size=batch_size)

        # 150 에포크 훈련
        for epoch in range(150):
            model.train()
            for data in train_loader:
                out = model(data.x, data.edge_index, data.batch, data.global_feat)
                loss = criterion(out, data.y)
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()

        # 검증
        model.eval()
        all_labels, all_probs = [], []
        with torch.no_grad():
            for data in val_loader:
                out = model(data.x, data.edge_index, data.batch, data.global_feat)
                probs = F.softmax(out, dim=1)[:, 1]
                all_labels.extend(data.y.tolist())
                all_probs.extend(probs.tolist())

        fold_aucs.append(roc_auc_score(all_labels, all_probs))

    return np.mean(fold_aucs)

# --- 🏃‍♂️ 5. Optuna Study 실행 ---
study = optuna.create_study(direction='maximize', study_name='gin_top3_feats_tuning')
study.optimize(objective, n_trials=50)

print("\n--- GIN 모델 + Top3 특징 Optuna 탐색 완료 ---")
print(f"최고의 평균 AUC: {study.best_value:.4f}")
print("최적의 하이퍼파라미터:")
print(study.best_params)

best_params_gin = study.best_params

In [ ]:
# --- ⚙️ 1. 라이브러리 임포트 ---
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
import numpy as np
import torch
from torch_geometric.data import Data
import os
from google.colab import drive

# --- ⚙️ 2. 기본 설정 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"

# ✨ 입력 파일: 손상이 말씀하신 고품질 데이터가 맞습니다.
INPUT_FILE = os.path.join(BASE_DIR, "pgp_dataset_inhibitor_vs_non.csv")
# ✨ 출력 파일: 최종적으로 사용할 그래프 데이터셋
OUTPUT_FILE = os.path.join(BASE_DIR, "pgp_graph_dataset_ultimate.pt")
# 사용할 상위 3개 특징 리스트
TOP_3_FEATURES = ['LogP', 'MolWt', 'TPSA']

# --- 🔬 3. 데이터 로딩 및 특징 공학 ---
print("Step 1: 고품질 데이터 로딩 및 특징 공학 적용...")
try:
    df = pd.read_csv(INPUT_FILE, sep='\t')
    print(f"  - 성공! '{os.path.basename(INPUT_FILE)}'에서 {len(df)}개 화합물 데이터를 로드했습니다.")
except Exception as e:
    print(f"  - 오류 발생: {e}")
    exit()

# 분자 기술자 계산 함수
def calculate_selected_descriptors(mol, selected_features):
    available_descriptors = {'MolWt': Descriptors.MolWt, 'LogP': Descriptors.MolLogP, 'TPSA': Descriptors.TPSA}
    return [available_descriptors[name](mol) for name in selected_features if name in available_descriptors]

descriptors_list = []
for smiles in df['smiles']:
    mol = Chem.MolFromSmiles(smiles) if pd.notna(smiles) else None
    if mol:
        descriptors_list.append(calculate_selected_descriptors(mol, TOP_3_FEATURES))
    else:
        descriptors_list.append([np.nan] * len(TOP_3_FEATURES))

df_descriptors = pd.DataFrame(descriptors_list, columns=TOP_3_FEATURES, index=df.index)
# 원본 데이터와 특징 데이터를 결합하고, 특징 계산에 실패한 행은 제거
final_df = pd.concat([df, df_descriptors], axis=1).dropna(subset=TOP_3_FEATURES)
print(f"  - 상위 3개 특징 추가 완료. 최종 데이터: {len(final_df)}개")


# --- 🧠 4. 최종 그래프 데이터셋 생성 ---
print("\nStep 2: 최종 그래프 데이터셋 생성...")

def get_atom_features(atom):
    return [float(f) for f in [atom.GetAtomicNum(), atom.GetDegree(), atom.GetFormalCharge(), atom.GetNumRadicalElectrons(), atom.GetHybridization(), atom.GetIsAromatic(), atom.IsInRing()]]

def smiles_to_graph(row, feature_names):
    mol = Chem.MolFromSmiles(row['smiles'])
    if not mol: return None
    atom_features = [get_atom_features(atom) for atom in mol.GetAtoms()]
    edge_indices = []
    for bond in mol.GetBonds():
        edge_indices.extend([[bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()], [bond.GetEndAtomIdx(), bond.GetBeginAtomIdx()]])
    global_features = row[feature_names].tolist()
    return Data(x=torch.tensor(atom_features, dtype=torch.float), edge_index=torch.tensor(edge_indices, dtype=torch.long).t().contiguous(),
                y=torch.tensor([row['pgp_inhibitor']], dtype=torch.long), global_feat=torch.tensor([global_features], dtype=torch.float))

dataset = [graph for index, row in final_df.iterrows() if (graph := smiles_to_graph(row, TOP_3_FEATURES)) is not None]

# 최종 그래프 데이터셋 저장
torch.save(dataset, OUTPUT_FILE)
print(f"  - {len(dataset)}개의 그래프 객체 생성 완료.")
print(f"\n--- ✅ 작업 완료! ---")
print(f"최종 그래프 데이터셋이 아래 경로에 저장되었습니다:\n{OUTPUT_FILE}")

In [ ]:
# --- ⚙️ 1. 라이브러리 임포트 ---
import pandas as pd
import torch
import torch.nn.functional as F
from torch.nn import Linear, BatchNorm1d, Dropout
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.loader import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np
import optuna
import os
from google.colab import drive

# --- ⚙️ 2. 설정 및 데이터 로딩 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
# ✨ 최종적으로 만든 '궁극의' 그래프 데이터셋을 불러옵니다.
GRAPH_DATASET_PATH = os.path.join(BASE_DIR, "pgp_graph_dataset_ultimate.pt")

print("GCN Model Test with Ultimate Data: Optuna 최적화 시작...")
try:
    dataset = torch.load(GRAPH_DATASET_PATH, weights_only=False)
    num_node_features = dataset[0].num_node_features
    num_global_features = dataset[0].global_feat.shape[1]
    print(f"  - 성공! {len(dataset)}개의 그래프 데이터를 로드했습니다.")
    print(f"  - 노드 특징 수: {num_node_features}, 전역 특징 수: {num_global_features}")
except Exception as e:
    print(f"  - 오류 발생: {e}")
    exit()

# --- 🧠 3. GCN 모델 정의 (Global Feature 결합) ---
class GCN_with_GlobalFeats(torch.nn.Module):
    def __init__(self, num_node_features, num_global_features, hidden_channels, dropout_rate):
        super(GCN_with_GlobalFeats, self).__init__()
        self.conv1 = GCNConv(num_node_features, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.combined_dim = hidden_channels + num_global_features
        self.fc1 = Linear(self.combined_dim, hidden_channels)
        self.bn1 = BatchNorm1d(hidden_channels)
        self.fc2 = Linear(hidden_channels, 2)
        self.dropout = Dropout(p=dropout_rate)

    def forward(self, x, edge_index, batch, global_feat):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        gnn_out = global_mean_pool(x, batch)
        combined = torch.cat([gnn_out, global_feat], dim=1)
        x = self.fc1(combined); x = self.bn1(x); x = F.relu(x); x = self.dropout(x)
        x = self.fc2(x)
        return x

# --- 🤖 4. Optuna 목적 함수 정의 ---
def objective(trial):
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    hidden_channels = trial.suggest_categorical('hidden_channels', [64, 128])
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.6)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    labels = np.array([data.y.item() for data in dataset])
    fold_aucs = []
    for train_idx, val_idx in skf.split(np.arange(len(dataset)), labels):
        model = GCN_with_GlobalFeats(num_node_features, num_global_features, hidden_channels, dropout_rate)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = torch.nn.CrossEntropyLoss()
        train_loader = DataLoader([dataset[i] for i in train_idx], batch_size=batch_size, shuffle=True)
        val_loader = DataLoader([dataset[i] for i in val_idx], batch_size=batch_size)
        for epoch in range(150):
            model.train()
            for data in train_loader:
                out = model(data.x, data.edge_index, data.batch, data.global_feat)
                loss = criterion(out, data.y); loss.backward(); optimizer.step(); optimizer.zero_grad()
        model.eval()
        all_labels, all_probs = [], []
        with torch.no_grad():
            for data in val_loader:
                out = model(data.x, data.edge_index, data.batch, data.global_feat)
                all_labels.extend(data.y.tolist()); all_probs.extend(F.softmax(out, dim=1)[:, 1].tolist())
        fold_aucs.append(roc_auc_score(all_labels, all_probs))
    return np.mean(fold_aucs)

# --- 🏃‍♂️ 5. Optuna Study 실행 ---
print("\nStep 3: GCN 모델 Optuna 최적화 시작...")
study = optuna.create_study(direction='maximize', study_name='gcn_ultimate_data_final')
study.optimize(objective, n_trials=50)

print("\n--- GCN 모델 + '궁극의 데이터' Optuna 탐색 완료 ---")
print(f"최고의 평균 AUC: {study.best_value:.4f}")
print("최적의 하이퍼파라미터:")
print(study.best_params)

In [ ]:
# --- ⚙️ 1. 라이브러리 임포트 ---
import pandas as pd
import torch
import torch.nn.functional as F
from torch.nn import Linear, BatchNorm1d, Dropout
# ✨ GATConv를 임포트합니다.
from torch_geometric.nn import GATConv, global_mean_pool
from torch_geometric.loader import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np
import optuna
import os
from google.colab import drive

# --- ⚙️ 2. 설정 및 데이터 로딩 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
# ✨ GCN 테스트 때와 동일한 '궁극의' 그래프 데이터셋을 불러옵니다.
GRAPH_DATASET_PATH = os.path.join(BASE_DIR, "pgp_graph_dataset_ultimate.pt")

print("GAT Model Test with Ultimate Data: Optuna 최적화 시작...")
try:
    dataset = torch.load(GRAPH_DATASET_PATH, weights_only=False)
    num_node_features = dataset[0].num_node_features
    num_global_features = dataset[0].global_feat.shape[1]
    print(f"  - 성공! {len(dataset)}개의 그래프 데이터를 로드했습니다.")
    print(f"  - 노드 특징 수: {num_node_features}, 전역 특징 수: {num_global_features}")
except Exception as e:
    print(f"  - 오류 발생: {e}")
    exit()

# --- 🧠 3. GAT 모델 정의 (Global Feature 결합) ---
class GAT_with_GlobalFeats(torch.nn.Module):
    def __init__(self, num_node_features, num_global_features, hidden_channels, dropout_rate, heads):
        super(GAT_with_GlobalFeats, self).__init__()
        # GAT의 핵심 파라미터인 어텐션 헤드(heads)를 사용합니다.
        self.conv1 = GATConv(num_node_features, hidden_channels, heads=heads)
        # Multi-head 출력을 합치므로, 다음 층의 입력 채널은 hidden_channels * heads가 됩니다.
        self.conv2 = GATConv(hidden_channels * heads, hidden_channels, heads=1)

        self.combined_dim = hidden_channels + num_global_features
        self.fc1 = Linear(self.combined_dim, hidden_channels)
        self.bn1 = BatchNorm1d(hidden_channels)
        self.fc2 = Linear(hidden_channels, 2)
        self.dropout = Dropout(p=dropout_rate)

    def forward(self, x, edge_index, batch, global_feat):
        x = self.conv1(x, edge_index).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index).relu()
        gnn_out = global_mean_pool(x, batch)
        combined = torch.cat([gnn_out, global_feat], dim=1)
        x = self.fc1(combined); x = self.bn1(x); x = F.relu(x); x = self.dropout(x)
        x = self.fc2(x)
        return x

# --- 🤖 4. Optuna 목적 함수 정의 ---
def objective(trial):
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    hidden_channels = trial.suggest_categorical('hidden_channels', [64, 128])
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.6)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
    # ✨ GAT를 위한 'heads' 파라미터를 탐색 범위에 추가합니다.
    heads = trial.suggest_categorical('heads', [2, 4, 8])

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    labels = np.array([data.y.item() for data in dataset])
    fold_aucs = []
    for train_idx, val_idx in skf.split(np.arange(len(dataset)), labels):
        # GAT 모델 초기화
        model = GAT_with_GlobalFeats(num_node_features, num_global_features, hidden_channels, dropout_rate, heads=heads)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = torch.nn.CrossEntropyLoss()

        train_loader = DataLoader([dataset[i] for i in train_idx], batch_size=batch_size, shuffle=True)
        val_loader = DataLoader([dataset[i] for i in val_idx], batch_size=batch_size)

        for epoch in range(150):
            model.train()
            for data in train_loader:
                out = model(data.x, data.edge_index, data.batch, data.global_feat)
                loss = criterion(out, data.y); loss.backward(); optimizer.step(); optimizer.zero_grad()

        model.eval()
        all_labels, all_probs = [], []
        with torch.no_grad():
            for data in val_loader:
                out = model(data.x, data.edge_index, data.batch, data.global_feat)
                all_labels.extend(data.y.tolist()); all_probs.extend(F.softmax(out, dim=1)[:, 1].tolist())
        fold_aucs.append(roc_auc_score(all_labels, all_probs))

    return np.mean(fold_aucs)

# --- 🏃‍♂️ 5. Optuna Study 실행 ---
print("\nStep 3: GAT 모델 Optuna 최적화 시작...")
study = optuna.create_study(direction='maximize', study_name='gat_ultimate_data_tuning')
study.optimize(objective, n_trials=50)

print("\n--- GAT 모델 + '궁극의 데이터' Optuna 탐색 완료 ---")
print(f"최고의 평균 AUC: {study.best_value:.4f}")
print("최적의 하이퍼파라미터:")
print(study.best_params)

In [ ]:
# --- ⚙️ 1. 라이브러리 임포트 ---
import pandas as pd
from sklearn.model_selection import StratifiedKFold
# 💡 RDKit은 더 이상 필요 없으므로 주석 처리하거나 삭제해도 됩니다.
# from rdkit import Chem
# from rdkit.Chem import Descriptors
import numpy as np
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU, BatchNorm1d, Dropout
from torch_geometric.data import Data
from torch_geometric.nn import GINConv, global_add_pool
from torch_geometric.loader import DataLoader
from sklearn.metrics import roc_auc_score
import optuna
import os
from google.colab import drive

# --- ⚙️ 2. 기본 설정 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
# TOP_3_FEATURES 변수는 더 이상 필요 없습니다.

# --- 🔬 3. 데이터 로딩 ---
# 💡✨ [변경점] GAT 코드와 동일하게 .pt 파일을 바로 로드합니다.
GRAPH_DATASET_PATH = os.path.join(BASE_DIR, "pgp_graph_dataset_ultimate.pt")
print("Step 1: 저장된 그래프 데이터셋 로딩...")

try:
    dataset = torch.load(GRAPH_DATASET_PATH, weights_only=False)
    num_node_features = dataset[0].num_node_features
    num_global_features = dataset[0].global_feat.shape[1]
    print(f"  - 성공! {len(dataset)}개의 그래프 데이터를 로드했습니다.")
    print(f"  - 노드 특징 수: {num_node_features}, 전역 특징 수: {num_global_features}")
except Exception as e:
    print(f"  - 오류 발생: {e}")
    exit()

# 💡✨ CSV로부터 그래프를 생성하던 아래 부분은 모두 삭제합니다.
# df = pd.read_csv(...)
# get_atom_features(...)
# smiles_to_graph(...)
# dataset = [graph for ...]

# --- 🧠 4. GIN 모델 정의 ---
# (이하 모델 정의, Optuna 목적 함수, Study 실행 코드는 모두 동일합니다)
class GIN_Model(torch.nn.Module):
    def __init__(self, num_node_features, num_global_features, hidden_channels, dropout_rate):
        super(GIN_Model, self).__init__()
        mlp1 = Sequential(Linear(num_node_features, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv1 = GINConv(mlp1)
        mlp2 = Sequential(Linear(hidden_channels, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv2 = GINConv(mlp2)

        self.combined_dim = hidden_channels + num_global_features
        self.fc1 = Linear(self.combined_dim, hidden_channels)
        self.bn1 = BatchNorm1d(hidden_channels)
        self.fc2 = Linear(hidden_channels, 2)
        self.dropout = Dropout(p=dropout_rate)

    def forward(self, x, edge_index, batch, global_feat):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        gnn_out = global_add_pool(x, batch)
        combined = torch.cat([gnn_out, global_feat], dim=1)
        x = self.fc1(combined); x = self.bn1(x); x = F.relu(x); x = self.dropout(x)
        x = self.fc2(x)
        return x

# --- 🤖 5. Optuna 목적 함수 정의 ---
def objective(trial):
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    hidden_channels = trial.suggest_categorical('hidden_channels', [64, 128])
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.6)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    labels = np.array([data.y.item() for data in dataset])
    fold_aucs = []

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    for train_idx, val_idx in skf.split(np.arange(len(dataset)), labels):
        model = GIN_Model(num_node_features, num_global_features, hidden_channels, dropout_rate).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = torch.nn.CrossEntropyLoss()
        train_loader = DataLoader([dataset[i] for i in train_idx], batch_size=batch_size, shuffle=True)
        val_loader = DataLoader([dataset[i] for i in val_idx], batch_size=batch_size)

        for epoch in range(150):
            model.train()
            for data in train_loader:
                data = data.to(device)
                optimizer.zero_grad()
                out = model(data.x, data.edge_index, data.batch, data.global_feat)
                loss = criterion(out, data.y)
                loss.backward()
                optimizer.step()

        model.eval()
        all_labels, all_probs = [], []
        with torch.no_grad():
            for data in val_loader:
                data = data.to(device)
                out = model(data.x, data.edge_index, data.batch, data.global_feat)
                all_labels.extend(data.y.cpu().tolist())
                all_probs.extend(F.softmax(out, dim=1).cpu()[:, 1].tolist())

        fold_aucs.append(roc_auc_score(all_labels, all_probs))

    return np.mean(fold_aucs)

# --- 🏃‍♂️ 6. Optuna Study 실행 ---
print("\nStep 2: GIN 모델 Optuna 최적화 시작...")
study = optuna.create_study(direction='maximize', study_name='gin_ultimate_data_tuning_from_pt')
study.optimize(objective, n_trials=50)

print("\n--- GIN 모델 + '궁극의 데이터' Optuna 탐색 완료 ---")
print(f"최고의 평균 AUC: {study.best_value:.4f}")
print("최적의 하이퍼파라미터:")
print(study.best_params)

In [ ]:
# --- ⚙️ 1. 라이브러리 임포트 ---
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
import numpy as np
import torch
import torch.nn.functional as F
from torch.nn import Linear, BatchNorm1d, Dropout
from torch_geometric.data import Data
from torch_geometric.nn import GATConv, global_mean_pool
from torch_geometric.loader import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import optuna
import os
from google.colab import drive

# --- ⚙️ 2. 기본 설정 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
INPUT_FILE = os.path.join(BASE_DIR, "pgp_final_dataset.csv") # '기질 vs 저해제' 데이터
# 손상께서 선택한 5개 특징
SELECTED_FEATURES = ['LogP', 'MolWt', 'TPSA', 'NumRotatableBonds', 'NumAromaticRings']

# --- 🔬 3. 데이터 준비 (특징 공학 + 그래프 변환) ---
print("Step 1: 데이터 로딩 및 그래프 생성 (5 Features)...")
# ✨✨✨ 구분자를 쉼표(,)로 수정했습니다. ✨✨✨
df = pd.read_csv(INPUT_FILE, sep=',')
df.dropna(subset=['smiles'], inplace=True)

def calculate_selected_descriptors(mol, feature_names):
    available_descriptors = {
        'MolWt': Descriptors.MolWt, 'LogP': Descriptors.MolLogP, 'TPSA': Descriptors.TPSA,
        'NumRotatableBonds': Descriptors.NumRotatableBonds, 'NumAromaticRings': Descriptors.NumAromaticRings
    }
    return [available_descriptors[name](mol) for name in feature_names]

descriptors_list = []
for smiles in df['smiles']:
    mol = Chem.MolFromSmiles(smiles) if pd.notna(smiles) else None
    if mol: descriptors_list.append(calculate_selected_descriptors(mol, SELECTED_FEATURES))
    else: descriptors_list.append([np.nan] * len(SELECTED_FEATURES))

df_descriptors = pd.DataFrame(descriptors_list, columns=SELECTED_FEATURES, index=df.index)
final_df = pd.concat([df, df_descriptors], axis=1).dropna(subset=SELECTED_FEATURES)

def get_atom_features(atom):
    return [float(f) for f in [atom.GetAtomicNum(), atom.GetDegree(), atom.GetFormalCharge(), atom.GetNumRadicalElectrons(), atom.GetHybridization(), atom.GetIsAromatic(), atom.IsInRing()]]

def smiles_to_graph(row, feature_names):
    mol = Chem.MolFromSmiles(row['smiles'])
    if not mol: return None
    atom_features = [get_atom_features(atom) for atom in mol.GetAtoms()]
    edge_indices = [[b.GetBeginAtomIdx(), b.GetEndAtomIdx()] for b in mol.GetBonds()]
    edge_indices.extend([[b[1], b[0]] for b in edge_indices])
    global_features = row[feature_names].tolist()
    return Data(x=torch.tensor(atom_features, dtype=torch.float), edge_index=torch.tensor(edge_indices, dtype=torch.long).t().contiguous(),
                y=torch.tensor([row['pgp_inhibitor']], dtype=torch.long), global_feat=torch.tensor([global_features], dtype=torch.float))

dataset = [g for _, row in final_df.iterrows() if (g := smiles_to_graph(row, SELECTED_FEATURES)) is not None]
num_node_features = dataset[0].num_node_features
num_global_features = dataset[0].global_feat.shape[1]
print(f"  - 그래프 데이터셋 생성 완료: {len(dataset)}개")

# --- 🧠 4. GAT 모델 정의 ---
class GAT_Model(torch.nn.Module):
    def __init__(self, num_node_features, num_global_features, hidden_channels, dropout_rate, heads):
        super(GAT_Model, self).__init__()
        self.conv1 = GATConv(num_node_features, hidden_channels, heads=heads)
        self.conv2 = GATConv(hidden_channels * heads, hidden_channels, heads=1)
        self.combined_dim = hidden_channels + num_global_features
        self.fc1 = Linear(self.combined_dim, hidden_channels); self.bn1 = BatchNorm1d(hidden_channels)
        self.fc2 = Linear(hidden_channels, 2); self.dropout = Dropout(p=dropout_rate)
    def forward(self, x, edge_index, batch, global_feat):
        x = self.conv1(x, edge_index).relu(); x = self.dropout(x)
        x = self.conv2(x, edge_index).relu(); gnn_out = global_mean_pool(x, batch)
        combined = torch.cat([gnn_out, global_feat], dim=1)
        x = self.fc1(combined); x = self.bn1(x); x = F.relu(x); x = self.dropout(x)
        x = self.fc2(x)
        return x

# --- 🤖 5. Optuna 최적화 ---
def objective(trial):
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    hidden_channels = trial.suggest_categorical('hidden_channels', [64, 128])
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.6)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
    heads = trial.suggest_categorical('heads', [2, 4, 8])

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    labels = np.array([data.y.item() for data in dataset])
    fold_aucs = []
    for train_idx, val_idx in skf.split(np.arange(len(dataset)), labels):
        model = GAT_Model(num_node_features, num_global_features, hidden_channels, dropout_rate, heads=heads)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = torch.nn.CrossEntropyLoss()
        train_loader = DataLoader([dataset[i] for i in train_idx], batch_size=batch_size, shuffle=True)
        val_loader = DataLoader([dataset[i] for i in val_idx], batch_size=batch_size)
        for epoch in range(150):
            model.train()
            for data in train_loader:
                out = model(data.x, data.edge_index, data.batch, data.global_feat)
                loss = criterion(out, data.y); loss.backward(); optimizer.step(); optimizer.zero_grad()
        model.eval()
        all_labels, all_probs = [], []
        with torch.no_grad():
            for data in val_loader:
                out = model(data.x, data.edge_index, data.batch, data.global_feat)
                all_labels.extend(data.y.tolist()); all_probs.extend(F.softmax(out, dim=1)[:, 1].tolist())
        fold_aucs.append(roc_auc_score(all_labels, all_probs))
    return np.mean(fold_aucs)

print("\nStep 2: GAT 모델 Optuna 최적화 시작...")
study_gat = optuna.create_study(direction='maximize', study_name='gat_5feats_final_tuning')
study_gat.optimize(objective, n_trials=50)

print("\n--- GAT 모델 + 5개 특징 Optuna 탐색 완료 ---")
print(f"최고의 평균 AUC: {study_gat.best_value:.4f}")
print("최적의 하이퍼파라미터:")
print(study_gat.best_params)

In [ ]:
!pip install "numpy<2.0" rdkit-pypi torch_geometric optuna -q

In [ ]:
# --- ⚙️ 1. 라이브러리 임포트 ---
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
import numpy as np
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU, BatchNorm1d, Dropout
from torch_geometric.data import Data
# ✨ GINConv와 global_add_pool을 임포트합니다.
from torch_geometric.nn import GINConv, global_add_pool
from torch_geometric.loader import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import optuna
import os
from google.colab import drive

# --- ⚙️ 2. 기본 설정 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
INPUT_FILE = os.path.join(BASE_DIR, "pgp_final_dataset.csv") # '기질 vs 저해제' 데이터
SELECTED_FEATURES = ['LogP', 'MolWt', 'TPSA', 'NumRotatableBonds', 'NumAromaticRings']

# --- 🔬 3. 데이터 준비 ---
print("Step 1: 데이터 로딩 및 그래프 생성 (5 Features)...")
df = pd.read_csv(INPUT_FILE, sep=',')
df.dropna(subset=['smiles'], inplace=True)

def calculate_selected_descriptors(mol, feature_names):
    available_descriptors = {
        'MolWt': Descriptors.MolWt, 'LogP': Descriptors.MolLogP, 'TPSA': Descriptors.TPSA,
        'NumRotatableBonds': Descriptors.NumRotatableBonds, 'NumAromaticRings': Descriptors.NumAromaticRings
    }
    return [available_descriptors[name](mol) for name in feature_names]

descriptors_list = []
for smiles in df['smiles']:
    mol = Chem.MolFromSmiles(smiles) if pd.notna(smiles) else None
    if mol: descriptors_list.append(calculate_selected_descriptors(mol, SELECTED_FEATURES))
    else: descriptors_list.append([np.nan] * len(SELECTED_FEATURES))

df_descriptors = pd.DataFrame(descriptors_list, columns=SELECTED_FEATURES, index=df.index)
final_df = pd.concat([df, df_descriptors], axis=1).dropna(subset=SELECTED_FEATURES)

def get_atom_features(atom):
    return [float(f) for f in [atom.GetAtomicNum(), atom.GetDegree(), atom.GetFormalCharge(), atom.GetNumRadicalElectrons(), atom.GetHybridization(), atom.GetIsAromatic(), atom.IsInRing()]]

def smiles_to_graph(row, feature_names):
    mol = Chem.MolFromSmiles(row['smiles'])
    if not mol: return None
    atom_features = [get_atom_features(atom) for atom in mol.GetAtoms()]
    edge_indices = [[b.GetBeginAtomIdx(), b.GetEndAtomIdx()] for b in mol.GetBonds()]
    edge_indices.extend([[b[1], b[0]] for b in edge_indices])
    global_features = row[feature_names].tolist()
    return Data(x=torch.tensor(atom_features, dtype=torch.float), edge_index=torch.tensor(edge_indices, dtype=torch.long).t().contiguous(),
                y=torch.tensor([row['pgp_inhibitor']], dtype=torch.long), global_feat=torch.tensor([global_features], dtype=torch.float))

dataset = [g for _, row in final_df.iterrows() if (g := smiles_to_graph(row, SELECTED_FEATURES)) is not None]
num_node_features = dataset[0].num_node_features
num_global_features = dataset[0].global_feat.shape[1]
print(f"  - 그래프 데이터셋 생성 완료: {len(dataset)}개")

# --- 🧠 4. GIN 모델 정의 ---
class GIN_Model(torch.nn.Module):
    def __init__(self, num_node_features, num_global_features, hidden_channels, dropout_rate):
        super(GIN_Model, self).__init__()
        mlp1 = Sequential(Linear(num_node_features, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv1 = GINConv(mlp1)
        mlp2 = Sequential(Linear(hidden_channels, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv2 = GINConv(mlp2)

        self.combined_dim = hidden_channels + num_global_features
        self.fc1 = Linear(self.combined_dim, hidden_channels); self.bn1 = BatchNorm1d(hidden_channels)
        self.fc2 = Linear(hidden_channels, 2); self.dropout = Dropout(p=dropout_rate)

    def forward(self, x, edge_index, batch, global_feat):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        gnn_out = global_add_pool(x, batch)
        combined = torch.cat([gnn_out, global_feat], dim=1)
        x = self.fc1(combined); x = self.bn1(x); x = F.relu(x); x = self.dropout(x)
        x = self.fc2(x)
        return x

# --- 🤖 5. Optuna 최적화 ---
def objective(trial):
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    hidden_channels = trial.suggest_categorical('hidden_channels', [64, 128])
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.6)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    labels = np.array([data.y.item() for data in dataset])
    fold_aucs = []
    for train_idx, val_idx in skf.split(np.arange(len(dataset)), labels):
        # GIN 모델 초기화
        model = GIN_Model(num_node_features, num_global_features, hidden_channels, dropout_rate)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = torch.nn.CrossEntropyLoss()
        train_loader = DataLoader([dataset[i] for i in train_idx], batch_size=batch_size, shuffle=True)
        val_loader = DataLoader([dataset[i] for i in val_idx], batch_size=batch_size)
        for epoch in range(150):
            model.train()
            for data in train_loader:
                out = model(data.x, data.edge_index, data.batch, data.global_feat)
                loss = criterion(out, data.y); loss.backward(); optimizer.step(); optimizer.zero_grad()
        model.eval()
        all_labels, all_probs = [], []
        with torch.no_grad():
            for data in val_loader:
                out = model(data.x, data.edge_index, data.batch, data.global_feat)
                all_labels.extend(data.y.tolist()); all_probs.extend(F.softmax(out, dim=1)[:, 1].tolist())
        fold_aucs.append(roc_auc_score(all_labels, all_probs))
    return np.mean(fold_aucs)

print("\nStep 2: GIN 모델 Optuna 최적화 시작...")
study_gin = optuna.create_study(direction='maximize', study_name='gin_5feats_final_tuning')
study_gin.optimize(objective, n_trials=50)

print("\n--- GIN 모델 + 5개 특징 Optuna 탐색 완료 ---")
print(f"최고의 평균 AUC: {study_gin.best_value:.4f}")
print("최적의 하이퍼파라미터:")
print(study_gin.best_params)

In [ ]:
## ⭐️ 오류 해결: rdkit, pandas 라이브러리를 먼저 설치합니다.
!pip install rdkit-pypi pandas -q
# ⭐️ 오류 해결 패키지 설치: NumPy 버전을 1.x대로 고정하고, 나머지 라이브러리를 설치합니다.
!pip install "numpy<2.0" -q
!pip install rdkit-pypi pandas torch_geometric -q
# --- ⚙️ 1. 라이브러리 임포트 ---
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
import numpy as np
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU, BatchNorm1d, Dropout
from torch_geometric.data import Data
import os
from google.colab import drive

# --- ⚙️ 2. 기본 설정 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
INPUT_FILE = os.path.join(BASE_DIR, "pgp_final_dataset.csv")
# ⭐️ 파일명 변경: 엣지 특징이 포함된 것을 명시
OUTPUT_FILE_WITH_EDGES = os.path.join(BASE_DIR, "pgp_graph_dataset_with_edges.pt")
# 사용할 3개 전역 특징 리스트
SELECTED_GLOBAL_FEATURES = ['LogP', 'MolWt', 'TPSA']

# --- 🔬 3. 데이터 준비 (전역 특징 + 엣지 특징 모두 포함) ---
print("Step 1: 최종 데이터셋 생성 시작 (3-Global feats + 4-Edge feats)...")
df = pd.read_csv(INPUT_FILE, sep=',')
df.dropna(subset=['smiles'], inplace=True)

# --- 특징 추출 함수 정의 ---
def get_atom_features(atom):
    return [float(f) for f in [atom.GetAtomicNum(), atom.GetDegree(), atom.GetFormalCharge(), atom.GetNumRadicalElectrons(), atom.GetHybridization(), atom.GetIsAromatic(), atom.IsInRing()]]

def get_bond_features(bond):
    return [float(f) for f in [bond.GetBondTypeAsDouble(), bond.GetIsAromatic(), bond.IsInRing(), bond.GetIsConjugated()]]

def calculate_global_descriptors(mol, feature_names):
    available_descriptors = {'MolWt': Descriptors.MolWt, 'LogP': Descriptors.MolLogP, 'TPSA': Descriptors.TPSA}
    return [available_descriptors[name](mol) for name in feature_names]

# 데이터프레임에 전역 특징 추가
descriptors_list = []
for smiles in df['smiles']:
    mol = Chem.MolFromSmiles(smiles) if pd.notna(smiles) else None
    if mol: descriptors_list.append(calculate_global_descriptors(mol, SELECTED_GLOBAL_FEATURES))
    else: descriptors_list.append([np.nan] * len(SELECTED_GLOBAL_FEATURES))
df_descriptors = pd.DataFrame(descriptors_list, columns=SELECTED_GLOBAL_FEATURES, index=df.index)
final_df = pd.concat([df, df_descriptors], axis=1).dropna(subset=SELECTED_GLOBAL_FEATURES)
print(f"  - 전역 특징 추가 완료. 총 {len(final_df)}개 화합물 처리.")

# --- 그래프 변환 함수 (엣지 특징 추가) ---
def smiles_to_graph_with_edges(row, feature_names):
    mol = Chem.MolFromSmiles(row['smiles'])
    if not mol: return None

    atom_features = [get_atom_features(atom) for atom in mol.GetAtoms()]

    edge_indices, edge_features = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bond_feat = get_bond_features(bond)
        edge_indices.extend([[i, j], [j, i]])
        edge_features.extend([bond_feat, bond_feat])

    global_features = row[feature_names].tolist()

    return Data(x=torch.tensor(atom_features, dtype=torch.float),
                edge_index=torch.tensor(edge_indices, dtype=torch.long).t().contiguous(),
                edge_attr=torch.tensor(edge_features, dtype=torch.float),
                y=torch.tensor([row['pgp_inhibitor']], dtype=torch.long),
                global_feat=torch.tensor([global_features], dtype=torch.float))

# 최종 데이터셋 생성 및 저장
dataset_with_edges = [g for _, row in final_df.iterrows() if (g := smiles_to_graph_with_edges(row, SELECTED_GLOBAL_FEATURES)) is not None]
# ⭐️ 변경된 파일명으로 저장
torch.save(dataset_with_edges, OUTPUT_FILE_WITH_EDGES)
print(f"  - 그래프 데이터셋 생성 완료: {len(dataset_with_edges)}개")
print(f"\n✅ 1단계 완료! 최종 데이터셋이 아래 경로에 저장되었습니다:\n{OUTPUT_FILE_WITH_EDGES}")

In [ ]:
# --- ⚙️ 라이브러리 임포트 ---
!pip install optuna -q
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU, BatchNorm1d, Dropout
from torch_geometric.loader import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np
import optuna
from torch_geometric.nn import GINEConv, global_add_pool

# --- 설정 및 데이터 로딩 ---
GRAPH_DATASET_PATH = "/content/drive/My Drive/PgpTextMining/pgp_graph_dataset_with_edges.pt"
print("GINEConv Model Optimization Run: 하이퍼파라미터 최적화 시작...")
try:
    dataset = torch.load(GRAPH_DATASET_PATH, weights_only=False)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    num_node_features = dataset[0].num_node_features
    num_edge_features = dataset[0].num_edge_features
    num_global_features = dataset[0].global_feat.shape[1]

    print(f"  - {len(dataset)}개의 그래프 데이터를 로드했습니다. ({device} 사용)")
    print(f"  - 노드 특징 수: {num_node_features}, 엣지 특징 수: {num_edge_features}, 전역 특징 수: {num_global_features}")

except FileNotFoundError:
    print(f"오류: '{GRAPH_DATASET_PATH}'를 찾을 수 없습니다. 1단계를 먼저 실행해주세요.")
    exit()

# --- 🧠 GINE 모델 정의 ---
class GINE_Model(torch.nn.Module):
    def __init__(self, num_node_features, num_edge_features, num_global_features, hidden_channels, dropout_rate):
        super(GINE_Model, self).__init__()

        mlp1 = Sequential(Linear(num_node_features, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv1 = GINEConv(mlp1, edge_dim=num_edge_features)

        mlp2 = Sequential(Linear(hidden_channels, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv2 = GINEConv(mlp2, edge_dim=num_edge_features)

        self.combined_dim = hidden_channels + num_global_features
        self.fc1 = Linear(self.combined_dim, hidden_channels)
        self.bn1 = BatchNorm1d(hidden_channels)
        self.fc2 = Linear(hidden_channels, 2)
        self.dropout = Dropout(p=dropout_rate)

    def forward(self, x, edge_index, batch, edge_attr, global_feat):
        x = self.conv1(x, edge_index, edge_attr=edge_attr).relu()
        x = self.conv2(x, edge_index, edge_attr=edge_attr).relu()
        gnn_out = global_add_pool(x, batch)
        combined = torch.cat([gnn_out, global_feat], dim=1)
        x = self.fc1(combined); x = self.bn1(x); x = F.relu(x); x = self.dropout(x)
        x = self.fc2(x)
        return x

# --- 🤖 Optuna 목적 함수 정의 ---
def objective(trial):
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    hidden_channels = trial.suggest_categorical('hidden_channels', [32, 64, 128])
    batch_size = trial.suggest_categorical('batch_size', [16, 32])
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.6)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    labels = np.array([data.y.item() for data in dataset])
    fold_aucs = []

    for train_idx, val_idx in skf.split(np.arange(len(dataset)), labels):
        model = GINE_Model(num_node_features, num_edge_features, num_global_features, hidden_channels, dropout_rate).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = torch.nn.CrossEntropyLoss()

        # ⭐️ 오류 해결: 마지막 배치가 1개일 경우를 대비해 drop_last=True 옵션 추가
        train_loader = DataLoader([dataset[i] for i in train_idx], batch_size=batch_size, shuffle=True, drop_last=True)
        val_loader = DataLoader([dataset[i] for i in val_idx], batch_size=batch_size, drop_last=True)

        for epoch in range(100):
            model.train()
            for data in train_loader:
                data = data.to(device)
                optimizer.zero_grad()
                out = model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
                loss = criterion(out, data.y)
                loss.backward()
                optimizer.step()

        model.eval()
        all_labels, all_probs = [], []
        with torch.no_grad():
            for data in val_loader:
                data = data.to(device)
                out = model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
                all_labels.extend(data.y.cpu().tolist())
                all_probs.extend(F.softmax(out, dim=1).cpu()[:, 1].tolist())

        # val_loader가 비어있을 경우를 대비한 방어 코드
        if not all_labels:
            print("Warning: Validation set was empty after drop_last=True. Skipping this fold.")
            continue

        fold_aucs.append(roc_auc_score(all_labels, all_probs))

    # 모든 fold가 스킵되었을 경우를 대비
    if not fold_aucs:
        return 0.0

    return np.mean(fold_aucs)

# --- 🏃‍♂️ Optuna Study 실행 ---
study = optuna.create_study(direction='maximize', study_name='gine_with_edges_5fold')
study.optimize(objective, n_trials=30)

print("\n--- GINE 모델 + 최종 데이터 Optuna 탐색 완료 ---")
print(f"최고의 평균 AUC: {study.best_value:.4f}")
print("최적의 하이퍼파라미터:")
print(study.best_params)

In [ ]:
# --- ⚙️ 라이브러리 설치 ---
# NumPy 버전을 1.x대로 고정하고, 나머지 라이브러리를 설치합니다.
!pip install "numpy<2.0" -q
!pip install rdkit-pypi pandas torch_geometric -q

# --- ⚙️ 1. 라이브러리 임포트 ---
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
import numpy as np
import torch
from torch_geometric.data import Data
import os
from google.colab import drive

# --- ⚙️ 2. 기본 설정 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
INPUT_FILE = os.path.join(BASE_DIR, "pgp_final_dataset.csv")
# ⭐️ 파일명 변경: 5개의 전역 특징이 포함된 것을 명시
OUTPUT_FILE_5FEATS = os.path.join(BASE_DIR, "pgp_graph_dataset_5feats_with_edges.pt")
# ⭐️ 사용할 5개 전역 특징 리스트
SELECTED_GLOBAL_FEATURES = ['LogP', 'MolWt', 'TPSA', 'NumRotatableBonds', 'NumAromaticRings']

# --- 🔬 3. 데이터 준비 ---
print("Step 1: 최종 데이터셋 생성 시작 (5-Global feats + 4-Edge feats)...")
df = pd.read_csv(INPUT_FILE, sep=',')
df.dropna(subset=['smiles'], inplace=True)

# --- 특징 추출 함수 정의 ---
def get_atom_features(atom):
    return [float(f) for f in [atom.GetAtomicNum(), atom.GetDegree(), atom.GetFormalCharge(), atom.GetNumRadicalElectrons(), atom.GetHybridization(), atom.GetIsAromatic(), atom.IsInRing()]]

def get_bond_features(bond):
    return [float(f) for f in [bond.GetBondTypeAsDouble(), bond.GetIsAromatic(), bond.IsInRing(), bond.GetIsConjugated()]]

# ⭐️ 5개 특징을 계산하도록 함수 수정
def calculate_global_descriptors(mol, feature_names):
    available_descriptors = {
        'MolWt': Descriptors.MolWt,
        'LogP': Descriptors.MolLogP,
        'TPSA': Descriptors.TPSA,
        'NumRotatableBonds': Descriptors.NumRotatableBonds,
        'NumAromaticRings': Descriptors.NumAromaticRings
    }
    return [available_descriptors[name](mol) for name in feature_names]

# 데이터프레임에 전역 특징 추가
descriptors_list = []
for smiles in df['smiles']:
    mol = Chem.MolFromSmiles(smiles) if pd.notna(smiles) else None
    if mol: descriptors_list.append(calculate_global_descriptors(mol, SELECTED_GLOBAL_FEATURES))
    else: descriptors_list.append([np.nan] * len(SELECTED_GLOBAL_FEATURES))
df_descriptors = pd.DataFrame(descriptors_list, columns=SELECTED_GLOBAL_FEATURES, index=df.index)
final_df = pd.concat([df, df_descriptors], axis=1).dropna(subset=SELECTED_GLOBAL_FEATURES)
print(f"  - 전역 특징 추가 완료. 총 {len(final_df)}개 화합물 처리.")

# --- 그래프 변환 함수 (이전과 동일) ---
def smiles_to_graph_with_edges(row, feature_names):
    mol = Chem.MolFromSmiles(row['smiles'])
    if not mol: return None
    atom_features = [get_atom_features(atom) for atom in mol.GetAtoms()]
    edge_indices, edge_features = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bond_feat = get_bond_features(bond)
        edge_indices.extend([[i, j], [j, i]])
        edge_features.extend([bond_feat, bond_feat])
    global_features = row[feature_names].tolist()
    return Data(x=torch.tensor(atom_features, dtype=torch.float),
                edge_index=torch.tensor(edge_indices, dtype=torch.long).t().contiguous(),
                edge_attr=torch.tensor(edge_features, dtype=torch.float),
                y=torch.tensor([row['pgp_inhibitor']], dtype=torch.long),
                global_feat=torch.tensor([global_features], dtype=torch.float))

# 최종 데이터셋 생성 및 저장
dataset_5feats = [g for _, row in final_df.iterrows() if (g := smiles_to_graph_with_edges(row, SELECTED_GLOBAL_FEATURES)) is not None]
torch.save(dataset_5feats, OUTPUT_FILE_5FEATS)
print(f"  - 그래프 데이터셋 생성 완료: {len(dataset_5feats)}개")
print(f"\n✅ 1단계 완료! 최종 데이터셋이 아래 경로에 저장되었습니다:\n{OUTPUT_FILE_5FEATS}")

In [ ]:
# --- ⚙️ 라이브러리 임포트 ---
!pip install optuna -q
# NumPy 버전은 세션 재시작 후에도 유지되지만, torch_geometric은 다시 설치해야 할 수 있습니다.
!pip install torch_geometric -q

import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU, BatchNorm1d, Dropout
from torch_geometric.loader import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np
import optuna
from torch_geometric.nn import GINEConv, global_add_pool

# --- 설정 및 데이터 로딩 ---
# ⭐️ 1단계에서 새로 생성한 '5개 특징' 데이터셋을 불러옵니다.
GRAPH_DATASET_PATH = "/content/drive/My Drive/PgpTextMining/pgp_graph_dataset_5feats_with_edges.pt"
print("GINEConv (5 Features) Model Optimization Run: 하이퍼파라미터 최적화 시작...")
try:
    dataset = torch.load(GRAPH_DATASET_PATH, weights_only=False)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    num_node_features = dataset[0].num_node_features
    num_edge_features = dataset[0].num_edge_features
    num_global_features = dataset[0].global_feat.shape[1]

    print(f"  - {len(dataset)}개의 그래프 데이터를 로드했습니다. ({device} 사용)")
    print(f"  - 노드 특징 수: {num_node_features}, 엣지 특징 수: {num_edge_features}, 전역 특징 수: {num_global_features}")

except FileNotFoundError:
    print(f"오류: '{GRAPH_DATASET_PATH}'를 찾을 수 없습니다. 1단계를 먼저 실행해주세요.")
    exit()

# --- 🧠 GINE 모델 정의 (이전과 동일, 유연하게 설계되어 수정 불필요) ---
class GINE_Model(torch.nn.Module):
    def __init__(self, num_node_features, num_edge_features, num_global_features, hidden_channels, dropout_rate):
        super(GINE_Model, self).__init__()
        mlp1 = Sequential(Linear(num_node_features, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv1 = GINEConv(mlp1, edge_dim=num_edge_features)
        mlp2 = Sequential(Linear(hidden_channels, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv2 = GINEConv(mlp2, edge_dim=num_edge_features)
        self.combined_dim = hidden_channels + num_global_features
        self.fc1 = Linear(self.combined_dim, hidden_channels)
        self.bn1 = BatchNorm1d(hidden_channels)
        self.fc2 = Linear(hidden_channels, 2)
        self.dropout = Dropout(p=dropout_rate)

    def forward(self, x, edge_index, batch, edge_attr, global_feat):
        x = self.conv1(x, edge_index, edge_attr=edge_attr).relu()
        x = self.conv2(x, edge_index, edge_attr=edge_attr).relu()
        gnn_out = global_add_pool(x, batch)
        combined = torch.cat([gnn_out, global_feat], dim=1)
        x = self.fc1(combined); x = self.bn1(x); x = F.relu(x); x = self.dropout(x)
        x = self.fc2(x)
        return x

# --- 🤖 Optuna 목적 함수 정의 (이전과 동일) ---
def objective(trial):
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    hidden_channels = trial.suggest_categorical('hidden_channels', [32, 64, 128])
    batch_size = trial.suggest_categorical('batch_size', [16, 32])
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.6)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    labels = np.array([data.y.item() for data in dataset])
    fold_aucs = []

    for train_idx, val_idx in skf.split(np.arange(len(dataset)), labels):
        model = GINE_Model(num_node_features, num_edge_features, num_global_features, hidden_channels, dropout_rate).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = torch.nn.CrossEntropyLoss()
        train_loader = DataLoader([dataset[i] for i in train_idx], batch_size=batch_size, shuffle=True, drop_last=True)
        val_loader = DataLoader([dataset[i] for i in val_idx], batch_size=batch_size, drop_last=True)

        for epoch in range(75):
            model.train()
            for data in train_loader:
                data = data.to(device)
                optimizer.zero_grad()
                out = model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
                loss = criterion(out, data.y)
                loss.backward()
                optimizer.step()

        model.eval()
        all_labels, all_probs = [], []
        with torch.no_grad():
            for data in val_loader:
                data = data.to(device)
                out = model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
                all_labels.extend(data.y.cpu().tolist())
                all_probs.extend(F.softmax(out, dim=1).cpu()[:, 1].tolist())

        if not all_labels: continue
        fold_aucs.append(roc_auc_score(all_labels, all_probs))

    if not fold_aucs: return 0.0
    return np.mean(fold_aucs)

# --- 🏃‍♂️ Optuna Study 실행 ---
# ⭐️ Study 이름 변경
study = optuna.create_study(direction='maximize', study_name='gine_5feats_5fold')
study.optimize(objective, n_trials=30)

print("\n--- GINE 모델 (5 Features) + 최종 데이터 Optuna 탐색 완료 ---")
print(f"최고의 평균 AUC: {study.best_value:.4f}")
print("최적의 하이퍼파라미터:")
print(study.best_params)

In [ ]:
# --- ⚙️ 라이브러리 임포트 ---
!pip install optuna -q
!pip install torch_geometric -q

import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU, BatchNorm1d, Dropout
from torch_geometric.loader import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np
import optuna
from torch_geometric.nn import GINEConv, global_add_pool
import matplotlib.pyplot as plt # ⭐️ 시각화를 위한 라이브러리

# --- 설정 및 데이터 로딩 ---
# ⭐️ 3개의 전역 특징과 엣지 특징이 모두 포함된 데이터셋 경로입니다.
GRAPH_DATASET_PATH = "/content/drive/My Drive/PgpTextMining/pgp_graph_dataset_with_edges.pt"
print("Final Analysis: GINEConv (3 Global Feats) + 5-Fold CV + Visualization")
try:
    dataset = torch.load(GRAPH_DATASET_PATH, weights_only=False)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    num_node_features = dataset[0].num_node_features
    num_edge_features = dataset[0].num_edge_features
    num_global_features = dataset[0].global_feat.shape[1]

    print(f"  - {len(dataset)}개의 그래프 데이터를 로드했습니다. ({device} 사용)")
    print(f"  - 노드 특징 수: {num_node_features}, 엣지 특징 수: {num_edge_features}, 전역 특징 수: {num_global_features}")

except FileNotFoundError:
    print(f"오류: '{GRAPH_DATASET_PATH}'를 찾을 수 없습니다. 데이터 생성 코드를 먼저 실행해주세요.")
    exit()

# --- 🧠 GINE 모델 정의 (수정 없음) ---
class GINE_Model(torch.nn.Module):
    def __init__(self, num_node_features, num_edge_features, num_global_features, hidden_channels, dropout_rate):
        super(GINE_Model, self).__init__()
        mlp1 = Sequential(Linear(num_node_features, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv1 = GINEConv(mlp1, edge_dim=num_edge_features)
        mlp2 = Sequential(Linear(hidden_channels, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv2 = GINEConv(mlp2, edge_dim=num_edge_features)
        self.combined_dim = hidden_channels + num_global_features
        self.fc1 = Linear(self.combined_dim, hidden_channels)
        self.bn1 = BatchNorm1d(hidden_channels)
        self.fc2 = Linear(hidden_channels, 2)
        self.dropout = Dropout(p=dropout_rate)

    def forward(self, x, edge_index, batch, edge_attr, global_feat):
        x = self.conv1(x, edge_index, edge_attr=edge_attr).relu()
        x = self.conv2(x, edge_index, edge_attr=edge_attr).relu()
        gnn_out = global_add_pool(x, batch)
        combined = torch.cat([gnn_out, global_feat], dim=1)
        x = self.fc1(combined); x = self.bn1(x); x = F.relu(x); x = self.dropout(x)
        x = self.fc2(x)
        return x

# --- 🤖 Optuna 목적 함수 (⭐️ 학습 곡선 기록 기능 추가) ---
def objective(trial):
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    hidden_channels = trial.suggest_categorical('hidden_channels', [32, 64, 128])
    batch_size = trial.suggest_categorical('batch_size', [16, 32])
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.6)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    labels = np.array([data.y.item() for data in dataset])
    fold_aucs = []

    # 마지막 Fold의 학습 기록을 저장하기 위한 변수
    last_fold_history = {}

    for fold, (train_idx, val_idx) in enumerate(skf.split(np.arange(len(dataset)), labels)):
        model = GINE_Model(num_node_features, num_edge_features, num_global_features, hidden_channels, dropout_rate).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = torch.nn.CrossEntropyLoss()
        train_loader = DataLoader([dataset[i] for i in train_idx], batch_size=batch_size, shuffle=True, drop_last=True)
        val_loader = DataLoader([dataset[i] for i in val_idx], batch_size=batch_size, drop_last=True)

        # 매 Epoch의 점수를 기록할 리스트
        train_losses, val_losses, val_aucs = [], [], []

        for epoch in range(75):
            model.train()
            epoch_train_loss = 0
            for data in train_loader:
                data = data.to(device)
                optimizer.zero_grad()
                out = model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
                loss = criterion(out, data.y)
                loss.backward()
                optimizer.step()
                epoch_train_loss += loss.item()
            train_losses.append(epoch_train_loss / len(train_loader))

            model.eval()
            epoch_val_loss = 0
            all_labels, all_probs = [], []
            with torch.no_grad():
                for data in val_loader:
                    data = data.to(device)
                    out = model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
                    loss = criterion(out, data.y)
                    epoch_val_loss += loss.item()
                    all_labels.extend(data.y.cpu().tolist())
                    all_probs.extend(F.softmax(out, dim=1).cpu()[:, 1].tolist())

            if not all_labels: continue
            val_losses.append(epoch_val_loss / len(val_loader))
            val_aucs.append(roc_auc_score(all_labels, all_probs))

        # 마지막 Fold의 history를 저장
        if fold == skf.get_n_splits() - 1:
            last_fold_history = {'train_loss': train_losses, 'val_loss': val_losses, 'val_auc': val_aucs}

        fold_aucs.append(val_aucs[-1]) # 마지막 epoch의 auc를 fold의 최종 점수로 기록

    trial.set_user_attr("history", last_fold_history)

    if not fold_aucs: return 0.0
    return np.mean(fold_aucs)

# --- 🏃‍♂️ Optuna Study 실행 ---
study = optuna.create_study(direction='maximize', study_name='gine_3feats_5fold_final')
study.optimize(objective, n_trials=30)

print("\n--- GINE 모델 (3 Features) + 최종 데이터 Optuna 탐색 완료 ---")
print(f"최고의 평균 AUC: {study.best_value:.4f}")
print("최적의 하이퍼파라미터:")
print(study.best_params)

# --- 📊 학습 곡선 시각화 ---
print("\n--- 최고 성능 Trial의 학습 곡선 ---")
# 최적의 성능을 보였던 Trial의 학습 기록 가져오기
best_history = study.best_trial.user_attrs["history"]

if best_history:
    plt.figure(figsize=(14, 6))

    # Loss 그래프
    plt.subplot(1, 2, 1)
    plt.plot(best_history['train_loss'], label='Train Loss', color='blue')
    plt.plot(best_history['val_loss'], label='Validation Loss', color='orange')
    plt.title('Loss Curve for Best Trial', fontsize=16)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.legend()
    plt.grid(True)

    # AUC 그래프
    plt.subplot(1, 2, 2)
    plt.plot(best_history['val_auc'], label='Validation AUC', color='green')
    # 가장 높은 AUC 지점 표시
    best_auc_epoch = np.argmax(best_history['val_auc'])
    best_auc_value = np.max(best_history['val_auc'])
    plt.scatter(best_auc_epoch, best_auc_value, color='red', zorder=5, label=f'Best AUC: {best_auc_value:.4f} at Epoch {best_auc_epoch+1}')
    plt.title('AUC Curve for Best Trial', fontsize=16)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('AUC', fontsize=12)
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()
else:
    print("학습 기록을 찾을 수 없습니다.")

In [ ]:
# --- 최종 Optuna 및 시각화 코드 셀 ---

if 'dataset' in locals() and dataset is not None:
    # --- 라이브러리 임포트 ---
    !pip install optuna -q
    !pip install torch_geometric -q

    import torch
    import torch.nn.functional as F
    from torch.nn import Linear, Sequential, ReLU, BatchNorm1d, Dropout
    from torch_geometric.loader import DataLoader
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score
    import numpy as np
    import optuna
    from torch_geometric.nn import GINEConv, global_add_pool
    import matplotlib.pyplot as plt

    # --- 전역 변수 설정 ---
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    num_node_features = dataset[0].num_node_features
    num_edge_features = dataset[0].num_edge_features
    num_global_features = dataset[0].global_feat.shape[1]

    # --- 모델 정의 (Dropout 적용) ---
    class GINE_Model(torch.nn.Module):
        def __init__(self, num_node_features, num_edge_features, num_global_features, hidden_channels, dropout_rate):
            super(GINE_Model, self).__init__()
            mlp1 = Sequential(Linear(num_node_features, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
            self.conv1 = GINEConv(mlp1, edge_dim=num_edge_features)
            mlp2 = Sequential(Linear(hidden_channels, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
            self.conv2 = GINEConv(mlp2, edge_dim=num_edge_features)
            self.combined_dim = hidden_channels + num_global_features
            self.fc1 = Linear(self.combined_dim, hidden_channels)
            self.bn1 = BatchNorm1d(hidden_channels)
            self.fc2 = Linear(hidden_channels, 2)
            self.dropout = Dropout(p=dropout_rate)

        def forward(self, x, edge_index, batch, edge_attr, global_feat):
            x = self.conv1(x, edge_index, edge_attr=edge_attr).relu()
            x = self.conv2(x, edge_index, edge_attr=edge_attr).relu()
            gnn_out = global_add_pool(x, batch)
            combined = torch.cat([gnn_out, global_feat], dim=1)
            x = self.fc1(combined); x = self.bn1(x); x = F.relu(x); x = self.dropout(x)
            x = self.fc2(x)
            return x

    # --- Optuna 목적 함수 ---
    def objective(trial):
        lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
        hidden_channels = trial.suggest_categorical('hidden_channels', [32, 64, 128])
        batch_size = trial.suggest_categorical('batch_size', [16, 32])
        dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        labels = np.array([data.y.item() for data in dataset])
        fold_aucs, last_fold_history = [], {}
        for fold, (train_idx, val_idx) in enumerate(skf.split(np.arange(len(dataset)), labels)):
            model = GINE_Model(num_node_features, num_edge_features, num_global_features, hidden_channels, dropout_rate).to(device)
            optimizer = torch.optim.Adam(model.parameters(), lr=lr)
            criterion = torch.nn.CrossEntropyLoss()
            train_loader = DataLoader([dataset[i] for i in train_idx], batch_size=batch_size, shuffle=True, drop_last=True)
            val_loader = DataLoader([dataset[i] for i in val_idx], batch_size=batch_size, drop_last=True)
            train_losses, val_losses, val_aucs = [], [], []
            for epoch in range(75):
                model.train()
                epoch_train_loss = 0
                for data in train_loader:
                    data = data.to(device)
                    optimizer.zero_grad()
                    out = model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
                    loss = criterion(out, data.y)
                    loss.backward()
                    optimizer.step()
                    epoch_train_loss += loss.item()
                train_losses.append(epoch_train_loss / len(train_loader))
                model.eval()
                epoch_val_loss = 0
                all_labels, all_probs = [], []
                with torch.no_grad():
                    for data in val_loader:
                        data = data.to(device)
                        out = model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
                        loss = criterion(out, data.y)
                        epoch_val_loss += loss.item()
                        all_labels.extend(data.y.cpu().tolist())
                        all_probs.extend(F.softmax(out, dim=1).cpu()[:, 1].tolist())
                if not all_labels: continue
                val_losses.append(epoch_val_loss / len(val_loader))
                val_aucs.append(roc_auc_score(all_labels, all_probs))
            if fold == skf.get_n_splits() - 1:
                last_fold_history = {'train_loss': train_losses, 'val_loss': val_losses, 'val_auc': val_aucs}
            fold_aucs.append(val_aucs[-1])
        trial.set_user_attr("history", last_fold_history)
        if not fold_aucs: return 0.0
        return np.mean(fold_aucs)

    # --- Optuna Study 실행 ---
    print("\n--- Optuna 최적화 실행 ---")
    study = optuna.create_study(direction='maximize', study_name='gine_3feats_dropout_final')
    study.optimize(objective, n_trials=30)
    print("\n--- [Dropout 적용] GINE 모델 Optuna 탐색 완료 ---")
    print(f"최고의 평균 AUC: {study.best_value:.4f}")
    print("최적의 하이퍼파라미터:")
    print(study.best_params)

    # --- 학습 곡선 시각화 ---
    print("\n--- 최고 성능 Trial의 학습 곡선 ---")
    best_history = study.best_trial.user_attrs.get("history", {})
    if best_history:
        plt.figure(figsize=(14, 6))
        plt.subplot(1, 2, 1)
        plt.plot(best_history['train_loss'], label='Train Loss', color='blue')
        plt.plot(best_history['val_loss'], label='Validation Loss', color='orange')
        plt.title('Loss Curve for Best Trial', fontsize=16)
        plt.xlabel('Epoch', fontsize=12)
        plt.ylabel('Loss', fontsize=12)
        plt.legend()
        plt.grid(True)
        plt.subplot(1, 2, 2)
        plt.plot(best_history['val_auc'], label='Validation AUC', color='green')
        best_auc_epoch = np.argmax(best_history['val_auc'])
        best_auc_value = np.max(best_history['val_auc'])
        plt.scatter(best_auc_epoch, best_auc_value, color='red', zorder=5, label=f'Best AUC: {best_auc_value:.4f} at Epoch {best_auc_epoch+1}')
        plt.title('AUC Curve for Best Trial', fontsize=16)
        plt.xlabel('Epoch', fontsize=12)
        plt.ylabel('AUC', fontsize=12)
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()
    else:
        print("학습 기록을 찾을 수 없습니다.")

else:
    print("\n데이터셋이 로드되지 않았습니다. Optuna 최적화를 건너뜁니다.")

In [ ]:
# --- 라이브러리 임포트 ---
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.loader import DataLoader

# --- 1. 데이터 분할 및 최적 파라미터 로드 ---
print("--- 최종 모델 훈련 및 평가 시작 ---")

# Optuna study에서 최적의 하이퍼파라미터 가져오기
best_params = study.best_params
print(f"✅ 최적 하이퍼파라미터 로드 완료: {best_params}")

# 전체 데이터를 훈련(80%) 및 테스트(20%) 세트로 분할
labels = np.array([data.y.item() for data in dataset])
train_dataset, test_dataset = train_test_split(dataset, test_size=0.2, random_state=42, stratify=labels)

print(f"  - 훈련 데이터: {len(train_dataset)}개")
print(f"  - 테스트 데이터: {len(test_dataset)}개")

# --- 2. 최적 파라미터로 모델 생성 및 훈련 ---
# **best_params로 모든 정보를 넘기는 대신, 모델 생성에 필요한 파라미터만 명시적으로 전달합니다.
final_model = GINE_Model(
    num_node_features=num_node_features,
    num_edge_features=num_edge_features,
    num_global_features=num_global_features,
    hidden_channels=best_params['hidden_channels'],
    dropout_rate=best_params['dropout_rate']
).to(device)

# optimizer와 loader를 만들 때는 best_params에서 각각 필요한 정보를 가져와 사용합니다.
optimizer = torch.optim.Adam(final_model.parameters(), lr=best_params['lr'])
criterion = torch.nn.CrossEntropyLoss()
train_loader = DataLoader(train_dataset, batch_size=best_params['batch_size'], shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=best_params['batch_size'], drop_last=True)

print(f"✅ 모델 생성 완료. 훈련을 시작합니다.")

# 이전 학습 곡선에서 확인한 최적의 epoch 근처로 훈련 (예: 70)
print("\n--- 모델 훈련 시작 (Epochs: 70) ---")
for epoch in range(70):
    final_model.train()
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = final_model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()

print("✅ 모델 훈련 완료!")

# --- 3. 테스트 데이터로 최종 성능 평가 ---
print("\n--- 최종 모델 성능 평가 (테스트 데이터) ---")
final_model.eval()
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for data in test_loader:
        data = data.to(device)
        out = final_model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
        preds = out.argmax(dim=1)
        probs = F.softmax(out, dim=1)[:, 1]

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(data.y.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

# --- 4. 최종 결과 리포트 ---
# 테스트 로더가 비어있을 경우를 대비한 방어 코드
if not all_labels:
    print("오류: 테스트 데이터가 없어 평가를 진행할 수 없습니다.")
else:
    test_auc = roc_auc_score(all_labels, all_probs)
    test_accuracy = accuracy_score(all_labels, all_preds)

    print(f"- Test Accuracy (정확도): {test_accuracy:.4f}")
    print(f"- Test AUC (Area Under Curve): {test_auc:.4f}")
    print("\n- Classification Report:")
    print(classification_report(all_labels, all_preds, target_names=['Non-Inhibitor', 'Inhibitor']))

    # Confusion Matrix 시각화
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Non-Inhibitor', 'Inhibitor'], yticklabels=['Non-Inhibitor', 'Inhibitor'])
    plt.title('Confusion Matrix')
    plt.ylabel('Actual Label')
    plt.xlabel('Predicted Label')
    plt.show()

In [ ]:
# --- 라이브러리 임포트 ---
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU, BatchNorm1d, Dropout
from torch_geometric.loader import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np
import optuna
from torch_geometric.nn import GINEConv, global_add_pool
import matplotlib.pyplot as plt
from google.colab import drive
import os # ✨✨✨ 이 줄이 추가되었습니다. ✨✨✨

# --- 설정 및 데이터 로딩 ---
drive.mount('/content/drive', force_remount=True)
BASE_DIR = "/content/drive/My Drive/PgpTextMining"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# ✨ 초기 접근법에서 사용했던, 엣지 특징이 포함된 데이터셋 경로입니다.
GRAPH_DATASET_PATH = os.path.join(BASE_DIR, "pgp_graph_dataset_with_edges.pt")

print("초기 접근법(Inhibitor vs Substrate) 모델 최적화 시작...")
try:
    dataset = torch.load(GRAPH_DATASET_PATH, weights_only=False)
    num_node_features = dataset[0].num_node_features
    num_edge_features = dataset[0].num_edge_features
    num_global_features = dataset[0].global_feat.shape[1]
    print(f"  - {len(dataset)}개의 그래프 데이터를 로드했습니다. ({DEVICE} 사용)")
except FileNotFoundError:
    print(f"오류: '{GRAPH_DATASET_PATH}'를 찾을 수 없습니다. 데이터 생성 코드를 먼저 실행해주세요.")
    exit()

# --- GINE 모델 정의 (수정 없음) ---
class GINE_Model(torch.nn.Module):
    def __init__(self, num_node_features, num_edge_features, num_global_features, hidden_channels, dropout_rate):
        super(GINE_Model, self).__init__()
        mlp1 = Sequential(Linear(num_node_features, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv1 = GINEConv(mlp1, edge_dim=num_edge_features)
        mlp2 = Sequential(Linear(hidden_channels, hidden_channels), BatchNorm1d(hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv2 = GINEConv(mlp2, edge_dim=num_edge_features)
        self.combined_dim = hidden_channels + num_global_features
        self.fc1 = Linear(self.combined_dim, hidden_channels)
        self.bn1 = BatchNorm1d(hidden_channels)
        self.fc2 = Linear(hidden_channels, 2)
        self.dropout = Dropout(p=dropout_rate)

    def forward(self, x, edge_index, batch, edge_attr, global_feat):
        x = self.conv1(x, edge_index, edge_attr=edge_attr).relu()
        x = self.conv2(x, edge_index, edge_attr=edge_attr).relu()
        gnn_out = global_add_pool(x, batch)
        combined = torch.cat([gnn_out, global_feat], dim=1)
        x = self.fc1(combined); x = self.bn1(x); x = F.relu(x); x = self.dropout(x)
        x = self.fc2(x)
        return x

# --- Optuna 목적 함수 (피드백 적용) ---
def objective(trial, dataset):
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    hidden_channels = trial.suggest_categorical('hidden_channels', [32, 64, 128])
    batch_size = trial.suggest_categorical('batch_size', [16, 32])
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.6)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    labels = np.array([data.y.item() for data in dataset])
    fold_val_losses, last_fold_history = [], {}

    for fold, (train_idx, val_idx) in enumerate(skf.split(np.arange(len(dataset)), labels), 1):
        model = GINE_Model(num_node_features, num_edge_features, num_global_features, hidden_channels, dropout_rate).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = torch.nn.CrossEntropyLoss()
        train_loader = DataLoader([dataset[i] for i in train_idx], batch_size=batch_size, shuffle=True, drop_last=True)
        val_loader = DataLoader([dataset[i] for i in val_idx], batch_size=batch_size, drop_last=True)
        train_losses, val_losses, val_aucs = [], [], []

        for epoch in range(75):
            model.train()
            epoch_train_loss = 0
            for data in train_loader:
                data = data.to(DEVICE); optimizer.zero_grad()
                out = model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
                loss = criterion(out, data.y); loss.backward(); optimizer.step()
                epoch_train_loss += loss.item()
            if len(train_loader) > 0: train_losses.append(epoch_train_loss / len(train_loader))

            model.eval()
            epoch_val_loss = 0
            all_labels, all_probs = [], []
            with torch.no_grad():
                for data in val_loader:
                    data = data.to(DEVICE)
                    out = model(data.x, data.edge_index, data.batch, data.edge_attr, data.global_feat)
                    loss = criterion(out, data.y)
                    epoch_val_loss += loss.item()
                    all_labels.extend(data.y.cpu().tolist())
                    all_probs.extend(F.softmax(out, dim=1).cpu()[:, 1].tolist())

            if not all_labels: continue
            if len(val_loader) > 0: val_losses.append(epoch_val_loss / len(val_loader))
            val_aucs.append(roc_auc_score(all_labels, all_probs))

        if val_losses: fold_val_losses.append(val_losses[-1])
        if fold == skf.get_n_splits():
            last_fold_history = {'train_loss': train_losses, 'val_loss': val_losses, 'val_auc': val_aucs}

    trial.set_user_attr("history", last_fold_history)
    if not fold_val_losses: return float('inf')
    return np.mean(fold_val_losses)

# --- Optuna Study 실행 (피드백 적용) ---
print("\n--- Optuna 최적화 실행 (목표: Validation Loss 최소화) ---")
study = optuna.create_study(direction='minimize', study_name='gine_initial_method_loss_min')
study.optimize(lambda trial: objective(trial, dataset), n_trials=30)

print("\n--- [Loss 최소화] 초기 접근법 모델 Optuna 탐색 완료 ---")
print(f"최저 평균 Validation Loss (5-fold CV): {study.best_value:.4f}")
print("최적의 하이퍼파라미터:")
print(study.best_params)

# --- 학습 곡선 시각화 ---
print("\n--- 최고 성능 Trial의 학습 곡선 ---")
best_history = study.best_trial.user_attrs.get("history", {})
if best_history and best_history.get('train_loss'):
    plt.figure(figsize=(14, 6))
    plt.subplot(1, 2, 1)
    plt.plot(best_history['train_loss'], label='Train Loss', color='blue')
    plt.plot(best_history['val_loss'], label='Validation Loss', color='orange')
    plt.title('Loss Curve for Best Trial', fontsize=16)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.legend(); plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(best_history['val_auc'], label='Validation AUC', color='green')
    best_auc_epoch = np.argmax(best_history['val_auc'])
    best_auc_value = np.max(best_history['val_auc'])
    plt.scatter(best_auc_epoch, best_auc_value, color='red', zorder=5, label=f'Best AUC: {best_auc_value:.4f} at Epoch {best_auc_epoch+1}')
    plt.title('AUC Curve for Best Trial', fontsize=16)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('AUC', fontsize=12)
    plt.legend(); plt.grid(True)

    plt.tight_layout(); plt.show()
else:
    print("학습 기록을 찾을 수 없습니다.")